# Depthwise Convolution weights for ConvNN Attention

In [31]:
import torch
import torch.nn as nn 
import torch.nn.functional as f 
import numpy as np
    
"""Multi-Head Self-Attention Implementation"""
class MultiHeadAttention(nn.Module): 
    def __init__(self, d_hidden, num_heads, attention_dropout):
        super(MultiHeadAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden
        self.num_heads = num_heads
        self.d_k = d_hidden // num_heads # dimension of each head
        self.dropout = nn.Dropout(attention_dropout)
        
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)        
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        attn_probs = self.dropout(torch.softmax(attn_scores, dim=-1))
        output = torch.matmul(attn_probs, V)
        return output, attn_probs
    
    def split_head(self, x): 
        batch_size, seq_length, d_hidden = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)
        
    def combine_heads(self, x): 
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden) 
    
    def forward(self, x, mask=None):
        q = self.split_head(self.W_q(x)) # (B, num_heads, seq_length, d_k)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))
        
        attn_output, _ = self.scaled_dot_product_attention(q, k, v, mask) # (B, num_heads, seq_length, d_k)
        output = self.W_o(self.combine_heads(attn_output)) # (B, seq_length, d_hidden)
        return output

In [32]:
import torch
import torch.nn as nn 
import torch.nn.functional as f 
import numpy as np

"""Regular ConvNN Attention Implementation"""
class MultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 convolution_type='depthwise',
                 seq_length=197):

        super(MultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        self.in_channels = d_hidden // num_heads
        self.out_channels = d_hidden // num_heads

        if convolution_type == 'standard': 
            self.conv = nn.Conv1d(
                in_channels=self.in_channels,
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                bias=False
            )
        elif convolution_type == 'depthwise':
            self.conv = nn.Conv1d(
                in_channels=self.in_channels, 
                out_channels=self.out_channels,
                kernel_size=self.K,
                stride=self.K,
                padding=0,
                groups=self.in_channels, 
                bias=False
            )
        elif convolution_type == 'depthwise-separable':
            self.conv = nn.Sequential(
                # Depthwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.in_channels,
                    kernel_size=self.K,
                    stride=self.K,
                    padding=0,
                    groups=self.in_channels,
                    bias=False
                ), 
                # Pointwise Convolution
                nn.Conv1d(
                    in_channels=self.in_channels,
                    out_channels=self.out_channels,
                    kernel_size=1,
                    stride=1,
                    padding=0, 
                    bias=False
                )
            )
        self.conv.weight.data.fill_(1.0)

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2) # (B, num_heads, seq_length, d_k)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size() 
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_hidden)

    def _prime(self, v, qk, K):
        # v: (B*num_heads, d_k, seq_length), qk: (B*num_heads, seq_length, seq_length)
        b, c, t = v.shape
        topk_values, topk_indices = torch.topk(qk, k=K, dim=2, largest=True)

        topk_values = torch.softmax(topk_values, dim=-1)
        topk_indices_exp = topk_indices.unsqueeze(1).expand(b, c, t, K)
        topk_values_exp = topk_values.unsqueeze(1).expand(b, c, t, K)

        v_expanded = v.unsqueeze(-1).expand(b, c, t, K).contiguous()
        prime = torch.gather(v_expanded, dim=2, index=topk_indices_exp)
        prime = topk_values_exp * prime
        prime = prime.view(b, c, -1)
        return prime

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL) - Q @ K^T
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Merge B and num_heads into dim for prime & conv 
        ## (B, NH, SL, DK) → (B*NH, DK, SL) for v and (B, NH, SL, SL) → (B*NH, SL, SL)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k).permute(0, 2, 1)
        am_merged = attn_matrix.reshape(B * self.num_heads, self.seq_length, self.seq_length)

        # Prime and Convolution 
        prime = self._prime(v_merged, am_merged, self.K)
        out = self.conv(prime) # (B*num_heads, d_k, seq_length) 

        # Reshape back: (B*NH, DK, SL) → (B, NH, SL, DK)
        out = out.permute(0, 2, 1).contiguous().view(B, self.num_heads, self.seq_length, self.d_k)
        out = self.dropout(out) 

        # Combine Heads and Final Linear Projection
        output = self.W_o(self.combine_heads(out)) # (B, SL, d_hidden)
        return output
        

In [33]:
import torch
import torch.nn as nn
import numpy as np
import triton
import triton.language as tl

# ==========================================
# 1. TRITON KERNEL (FORWARD PASS)
# ==========================================
@triton.jit
def fused_prime_conv_fwd_kernel(
    # Pointers to matrices
    v_ptr, indices_ptr, values_ptr, weight_ptr, out_ptr,
    # Strides to handle memory layout
    stride_vb, stride_vt, stride_vd,
    stride_ib, stride_it, stride_ik,
    stride_wb, stride_wk,
    stride_ob, stride_ot, stride_od,
    # Matrix dimensions
    B_NH, T, D: tl.constexpr, K: tl.constexpr,
    # Meta-parameters
    BLOCK_D: tl.constexpr
):
    """
    Fuses the gathering of V, multiplication by Top-K attention weights, 
    and the depthwise convolution step.
    """
    pid_b_t = tl.program_id(0) # 1D grid covering Batch*Heads and Seq_len
    pid_b = pid_b_t // T
    pid_t = pid_b_t % T

    # Set up channel offsets
    d_offsets = tl.arange(0, BLOCK_D)
    mask_d = d_offsets < D

    # Initialize accumulator for the convolution sum
    acc = tl.zeros([BLOCK_D], dtype=tl.float32)

    # Loop over the Top-K elements
    for k in range(K):
        # 1. Load the index and attention value for the k-th top element
        idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
        v_idx = tl.load(indices_ptr + idx_offset)
        attn_val = tl.load(values_ptr + idx_offset)

        # 2. Load the V vector for all D channels at the gathered index
        v_offsets = pid_b * stride_vb + v_idx * stride_vt + d_offsets * stride_vd
        v_vec = tl.load(v_ptr + v_offsets, mask=mask_d, other=0.0)

        # 3. Load the Depthwise Convolution weight for this K step
        w_offsets = d_offsets * stride_wb + k * stride_wk
        w_vec = tl.load(weight_ptr + w_offsets, mask=mask_d, other=0.0)

        # 4. Multiply and accumulate (Gather * Attn_Value * Conv_Weight)
        acc += v_vec * attn_val * w_vec

    # Store the final convolved output
    out_offsets = pid_b * stride_ob + pid_t * stride_ot + d_offsets * stride_od
    tl.store(out_ptr + out_offsets, acc, mask=mask_d)


# ==========================================
# 2. AUTOGRAD WRAPPER (FORWARD + BACKWARD)
# ==========================================
class FusedPrimeConvFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        # Save tensors needed for the backward pass
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        
        # Ensure contiguous memory for predictable strides
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.squeeze(1).contiguous() # Squeeze from (D, 1, K) to (D, K)
        
        out = torch.empty_like(v)
        
        # Grid computes one block per query token per batch/head
        grid = lambda meta: (B_NH * T, )
        BLOCK_D = triton.next_power_of_2(D)
        
        fused_prime_conv_fwd_kernel[grid](
            v, topk_indices, topk_values, weight, out,
            v.stride(0), v.stride(1), v.stride(2),
            topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
            weight.stride(0), weight.stride(1),
            out.stride(0), out.stride(1), out.stride(2),
            B_NH, T, D, K,
            BLOCK_D=BLOCK_D
        )
        return out

    @staticmethod
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) # (D, K)
        
        grad_out = grad_out.contiguous()

        # Flatten indices to (B_NH, T*K, 1) and expand to D channels
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        
        # Gather V directly to shape (B_NH, T*K, D), then reshape
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        # Pre-compute shapes for broadcasting
        grad_out_exp = grad_out.unsqueeze(2)                # (B_NH, T, 1, D)
        val_exp = topk_values.unsqueeze(-1)                 # (B_NH, T, K, 1)
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0) # (1, 1, K, D)

        # Gradient w.r.t topk_values (dVal)
        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1) # (B_NH, T, K)

        # Gradient w.r.t conv_weight (dW)
        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) # (K, D)
        grad_weight = grad_weight_raw.t().unsqueeze(1) # Transpose & unsqueeze back to (D, 1, K)

        # Gradient w.r.t V (dV)
        dv_gathered = grad_out_exp * val_exp * weight_t_exp # (B_NH, T, K, D)
        grad_v = torch.zeros_like(v)
        
        # Scatter add the gradients back to the original V locations
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        # topk_indices is discrete, so its gradient is None.
        return grad_v, None, grad_val, grad_weight



######### NEW backward kernel code but resulted in slower speed ###########
# @triton.jit
# def fused_prime_conv_bwd_kernel(
#     # Pointers
#     grad_out_ptr, v_ptr, indices_ptr, values_ptr, weight_ptr,
#     grad_v_ptr, grad_val_ptr, grad_weight_ptr,
#     # Strides
#     stride_gob, stride_got, stride_god,
#     stride_vb, stride_vt, stride_vd,
#     stride_ib, stride_it, stride_ik,
#     stride_wb, stride_wk,
#     # Dimensions
#     B_NH, T, D: tl.constexpr, K: tl.constexpr,
#     BLOCK_D: tl.constexpr
# ):
#     """
#     Fuses the backward pass, calculating dV, dVal, and dW entirely in SRAM
#     and safely scattering the results back using atomic adds.
#     """
#     pid = tl.program_id(0)
#     pid_b = pid // T
#     pid_t = pid % T

#     d_offsets = tl.arange(0, BLOCK_D)
#     mask_d = d_offsets < D

#     # 1. Load the incoming gradient (dO) for this specific query token
#     go_offsets = pid_b * stride_gob + pid_t * stride_got + d_offsets * stride_god
#     go_vec = tl.load(grad_out_ptr + go_offsets, mask=mask_d, other=0.0)

#     # Loop over the Top-K elements
#     for k in range(K):
#         # Load the gathered index and the attention value
#         idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
#         idx = tl.load(indices_ptr + idx_offset)
#         val = tl.load(values_ptr + idx_offset)

#         # Load the original V vector at the gathered index
#         v_offsets = pid_b * stride_vb + idx * stride_vt + d_offsets * stride_vd
#         v_vec = tl.load(v_ptr + v_offsets, mask=mask_d, other=0.0)

#         # Load the Convolution Weight for this K step
#         w_offsets = d_offsets * stride_wb + k * stride_wk
#         w_vec = tl.load(weight_ptr + w_offsets, mask=mask_d, other=0.0)

#         # --------------------------------------------------
#         # GRADIENT CALCULATIONS & ATOMIC SCATTERS
#         # --------------------------------------------------

#         # 1. Gradient w.r.t Top-K Values (dVal)
#         # Math: sum_d(dO * V * W)
#         g_val_vec = go_vec * v_vec * w_vec
#         g_val = tl.sum(g_val_vec, axis=0)
#         tl.store(grad_val_ptr + idx_offset, g_val)

#         # 2. Gradient w.r.t V (dV)
#         # Math: dO * val * W 
#         # (Must use atomic_add because multiple queries might attend to the same V index)
#         g_v_update = go_vec * val * w_vec
#         tl.atomic_add(grad_v_ptr + v_offsets, g_v_update, mask=mask_d)

#         # 3. Gradient w.r.t Convolution Weights (dW)
#         # Math: dO * V * val
#         # (Must use atomic_add because all threads update the same global weights)
#         g_w_update = go_vec * v_vec * val
#         tl.atomic_add(grad_weight_ptr + w_offsets, g_w_update, mask=mask_d)

# class FusedPrimeConvFunction(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, v, topk_indices, topk_values, conv_weight):
#         ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
#         B_NH, T, D = v.shape
#         _, _, K = topk_indices.shape
        
#         v = v.contiguous()
#         topk_indices = topk_indices.contiguous()
#         topk_values = topk_values.contiguous()
#         weight = conv_weight.squeeze(1).contiguous()
        
#         out = torch.empty_like(v)
#         grid = lambda meta: (B_NH * T, )
#         BLOCK_D = triton.next_power_of_2(D)
        
#         fused_prime_conv_fwd_kernel[grid](
#             v, topk_indices, topk_values, weight, out,
#             v.stride(0), v.stride(1), v.stride(2),
#             topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
#             weight.stride(0), weight.stride(1),
#             out.stride(0), out.stride(1), out.stride(2),
#             B_NH, T, D, K,
#             BLOCK_D=BLOCK_D
#         )
#         return out

#     @staticmethod
#     def backward(ctx, grad_out):
#         v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
#         B_NH, T, D = v.shape
#         _, _, K = topk_indices.shape
#         weight = conv_weight.squeeze(1).contiguous()
        
#         grad_out = grad_out.contiguous()

#         # Initialize output gradient tensors
#         # grad_v and grad_weight MUST be zeroed out because we atomic_add into them
#         grad_v = torch.zeros_like(v)
#         grad_val = torch.empty_like(topk_values) # Doesn't need zeros, we do a direct store
#         grad_weight = torch.zeros_like(weight)
        
#         grid = lambda meta: (B_NH * T, )
#         BLOCK_D = triton.next_power_of_2(D)

#         fused_prime_conv_bwd_kernel[grid](
#             grad_out, v, topk_indices, topk_values, weight,
#             grad_v, grad_val, grad_weight,
#             grad_out.stride(0), grad_out.stride(1), grad_out.stride(2),
#             v.stride(0), v.stride(1), v.stride(2),
#             topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
#             weight.stride(0), weight.stride(1),
#             B_NH, T, D, K,
#             BLOCK_D=BLOCK_D
#         )

#         # Reshape grad_weight back to PyTorch's expected Depthwise Conv1d shape: (D, 1, K)
#         return grad_v, None, grad_val, grad_weight.unsqueeze(1)


# ==========================================
# 3. PYTORCH MODULE
# ==========================================
class FastMultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 seq_length=197):

        super(FastMultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        # Depthwise Convolution weights matching PyTorch's native Conv1d shape
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL)
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Top-K Selection
        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        # Merge Batch and Heads for the Triton Kernel
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # Apply Fused Triton Operation (Forward + Backward handled automatically)
        out = FusedPrimeConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )

        # Reshape back: (B*NH, SL, DK) → (B, NH, SL, DK) → (B, SL, d_hidden)
        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        # Final Linear Projection
        output = self.W_o(out)
        return output


# ==========================================
# 4. QUICK VERIFICATION TEST
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if device.type != "cuda":
        print("Warning: Triton requires a CUDA-enabled GPU. This test will fail on CPU.")
    else:
        # Hyperparameters matching standard ViT-Base
        BATCH_SIZE = 2
        SEQ_LENGTH = 197
        D_HIDDEN = 768
        NUM_HEADS = 12
        K = 8
        
        print("Initializing FastMultiHeadConvNNAttention...")
        model = FastMultiHeadConvNNAttention(
            d_hidden=D_HIDDEN, 
            num_heads=NUM_HEADS, 
            attention_dropout=0.1, 
            K=K, 
            seq_length=SEQ_LENGTH
        ).to(device)
        
        # Dummy input
        x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)
        
        print("Running Forward Pass...")
        out = model(x)
        print(f"Output shape: {out.shape} (Expected: [{BATCH_SIZE}, {SEQ_LENGTH}, {D_HIDDEN}])")
        
        print("Running Backward Pass...")
        loss = out.sum()
        loss.backward()
        print(f"Input gradient shape: {x.grad.shape}")
        print("Success! Forward and backward passes completed.")

Initializing FastMultiHeadConvNNAttention...
Running Forward Pass...
Output shape: torch.Size([2, 197, 768]) (Expected: [2, 197, 768])
Running Backward Pass...
Input gradient shape: torch.Size([2, 197, 768])
Success! Forward and backward passes completed.


## Speed Test

In [34]:
import torch
import numpy as np

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above this)

def benchmark_module(module, x, num_iters=100):
    # 1. Warm-up
    # GPUs have initialization overhead. We run a few dummy passes first.
    for _ in range(10):
        out = module(x)
        loss = out.sum()
        loss.backward()
    
    torch.cuda.synchronize()
    
    # 2. Memory Benchmark
    # Forward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    out = module(x)
    fwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # Backward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    loss = out.sum()
    loss.backward()
    bwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # 3. Speed Benchmark using CUDA Events
    fwd_times = []
    bwd_times = []
    
    for _ in range(num_iters):
        # Time Forward
        torch.cuda.synchronize()
        start_fwd = torch.cuda.Event(enable_timing=True)
        end_fwd = torch.cuda.Event(enable_timing=True)
        
        start_fwd.record()
        out = module(x)
        end_fwd.record()
        torch.cuda.synchronize()
        fwd_times.append(start_fwd.elapsed_time(end_fwd))
        
        # Time Backward
        loss = out.sum()
        torch.cuda.synchronize()
        start_bwd = torch.cuda.Event(enable_timing=True)
        end_bwd = torch.cuda.Event(enable_timing=True)
        
        start_bwd.record()
        loss.backward()
        end_bwd.record()
        torch.cuda.synchronize()
        bwd_times.append(start_bwd.elapsed_time(end_bwd))
        
    avg_fwd = np.mean(fwd_times)
    avg_bwd = np.mean(bwd_times)
    
    return avg_fwd, avg_bwd, fwd_mem, bwd_mem

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Benchmarking requires a CUDA GPU.")

    # Standard ViT parameters
    BATCH_SIZE = 32
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 9
    
    print(f"Benchmarking with Batch Size: {BATCH_SIZE}, Seq Length: {SEQ_LENGTH}")
    print("-" * 60)

    # Initialize Modules

    standard_model = MultiHeadAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0
    ).to(device)
    
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # We reuse the exact same input tensor to ensure fairness
    x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)

    print("Benchmarking Standard Multi-Head Attention...")
    std_fwd, std_bwd, std_fmem, std_bmem = benchmark_module(standard_model, x)
    
    print("Benchmarking Original Implementation...")
    old_fwd, old_bwd, old_fmem, old_bmem = benchmark_module(old_model, x)

    print("Benchmarking Triton Implementation...")
    new_fwd, new_bwd, new_fmem, new_bmem = benchmark_module(new_model, x)

    # Print Results Table
    print("\n" + "=" * 85)
    print("Normalized to Original Implementation")
    print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'Improvement'}")
    print("-" * 85)
    print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {new_fwd:<12.2f} | {old_fwd/new_fwd:.2f}x faster")
    print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {new_bwd:<12.2f} | {old_bwd/new_bwd:.2f}x faster")
    print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {new_fmem:<12.2f} | {old_fmem/new_fmem:.2f}x less")
    print(f"{'Backward Peak Memory (MB)':<25} | {std_bmem:<12.2f} | {old_bmem:<12.2f} | {new_bmem:<12.2f} | {old_bmem/new_bmem:.2f}x less")
    print("=" * 85)

    print("\n" + "=" * 115)
    print("Normalized to Standard Multi-Head Attention")
    print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'Original vs Standard'} | {'Triton vs Standard'}")
    print("-" * 115)
    print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {new_fwd:<12.2f} | {old_fwd/std_fwd:.2f}x slower         | {new_fwd/std_fwd:.2f}x slower")
    print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {new_bwd:<12.2f} | {old_bwd/std_bwd:.2f}x slower         | {new_bwd/std_bwd:.2f}x slower")
    print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {new_fmem:<12.2f} | {old_fmem/std_fmem:.2f}x more           | {new_fmem/std_fmem:.2f}x more")
    print(f"{'Backward Peak Memory (MB)':<25} | {std_bmem:<12.2f} | {old_bmem:<12.2f} | {new_bmem:<12.2f} | {old_bmem/std_bmem:.2f}x more           | {new_bmem/std_bmem:.2f}x more")
    print("=" * 115)
    

Benchmarking with Batch Size: 32, Seq Length: 197
------------------------------------------------------------
Benchmarking Standard Multi-Head Attention...
Benchmarking Original Implementation...
Benchmarking Triton Implementation...

Normalized to Original Implementation
Metric                    | Standard     | Original     | Triton       | Improvement
-------------------------------------------------------------------------------------
Forward Time (ms)         | 2.58         | 4.51         | 3.72         | 1.21x faster
Backward Time (ms)        | 5.61         | 9.73         | 7.07         | 1.38x faster
Forward Peak Memory (MB)  | 575.44       | 1071.33      | 563.56       | 1.90x less
Backward Peak Memory (MB) | 614.92       | 1217.29      | 934.09       | 1.30x less

Normalized to Standard Multi-Head Attention
Metric                    | Standard     | Original     | Triton       | Original vs Standard | Triton vs Standard
-------------------------------------------------------

## Sanity Check for Output and Gradient Calculations

In [35]:
import torch

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above)

def verify_correctness():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='depthwise', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to Triton model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Triton implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to Triton model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000000

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000015
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000286
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00000238

SUCCESS: Triton implementation is mathematically equivalent!


# Standard Convolution weights for ConvNN Attention

In [36]:
"""
FastConvNNAttention.py

- Implementation of ConvNNAttention with STANDARD (Regular) Convolution.
- Fuses the gathering of V, multiplication by Top-K attention weights, and the convolution step.
"""

import torch
import torch.nn as nn
import numpy as np
import triton
import triton.language as tl
from torch.amp import custom_fwd, custom_bwd

# ==========================================
# 1. TRITON KERNEL (FORWARD PASS)
# ==========================================
@triton.jit
def fused_prime_standard_conv_fwd_kernel(
    # Pointers to matrices
    v_ptr, indices_ptr, values_ptr, weight_ptr, out_ptr,
    # Strides to handle memory layout
    stride_vb, stride_vt, stride_vd,
    stride_ib, stride_it, stride_ik,
    stride_w_dout, stride_w_din, stride_w_k, # 3 Strides for (D_out, D_in, K)
    stride_ob, stride_ot, stride_od,
    # Matrix dimensions
    B_NH, T, D: tl.constexpr, K: tl.constexpr,
    # Meta-parameters
    BLOCK_D: tl.constexpr
):
    """
    Fuses the gathering of V, multiplication by Top-K attention weights, 
    and a STANDARD 1D convolution step across all channels.
    """
    pid_b_t = tl.program_id(0) # 1D grid covering Batch*Heads and Seq_len
    pid_b = pid_b_t // T
    pid_t = pid_b_t % T

    # Set up channel offsets for both output (d_out) and input (d_in) dimensions
    d_out = tl.arange(0, BLOCK_D)
    d_in = tl.arange(0, BLOCK_D)
    mask_out = d_out < D
    mask_in = d_in < D

    # Initialize accumulator for the output channels
    acc = tl.zeros([BLOCK_D], dtype=tl.float32)

    # Loop over the Top-K elements
    for k in range(K):
        # 1. Load the index and attention value for the k-th top element
        idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
        v_idx = tl.load(indices_ptr + idx_offset)
        attn_val = tl.load(values_ptr + idx_offset)

        # 2. Load the Input V vector for all D_in channels at the gathered index
        v_offsets = pid_b * stride_vb + v_idx * stride_vt + d_in * stride_vd
        v_vec = tl.load(v_ptr + v_offsets, mask=mask_in, other=0.0) # Shape: (D_in,)

        # 3. Load the Standard Convolution weight slice for this K step
        w_offsets = d_out[:, None] * stride_w_dout + d_in[None, :] * stride_w_din + k * stride_w_k
        w_slice = tl.load(weight_ptr + w_offsets, mask=(mask_out[:, None] & mask_in[None, :]), other=0.0) # Shape: (D_out, D_in)

        # 4. Multiply and accumulate (Matrix-Vector Product: W @ V * Attn)
        # Broadcasting v_vec across d_out handles the cross-channel mixing
        prod = w_slice * v_vec[None, :] * attn_val
        acc += tl.sum(prod, axis=1) # Sum over D_in (axis 1)
        
    # Cast the fp32 accumulator back to the pointer's original dtype (fp16, bf16, or fp32)
    acc_casted = acc.to(v_ptr.dtype.element_ty) 
    
    # Store the final convolved output
    out_offsets = pid_b * stride_ob + pid_t * stride_ot + d_out * stride_od
    tl.store(out_ptr + out_offsets, acc_casted, mask=mask_out)


# ==========================================
# 2. AUTOGRAD WRAPPER (FORWARD + BACKWARD)
# ==========================================
class FusedPrimeConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda')
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        
        # Ensure contiguous memory for predictable strides
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.contiguous() # (D_out, D_in, K) - No squeezing required!
        
        out = torch.empty_like(v)
        
        # Grid computes one block per query token per batch/head
        grid = lambda meta: (B_NH * T, )
        BLOCK_D = triton.next_power_of_2(D)
        
        fused_prime_standard_conv_fwd_kernel[grid](
            v, topk_indices, topk_values, weight, out,
            v.stride(0), v.stride(1), v.stride(2),
            topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
            weight.stride(0), weight.stride(1), weight.stride(2), # 3 Strides mapped
            out.stride(0), out.stride(1), out.stride(2),
            B_NH, T, D, K,
            BLOCK_D=BLOCK_D
        )
        return out

    @staticmethod
    @custom_bwd(device_type='cuda') 
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight # (D_out, D_in, K)
        
        grad_out = grad_out.contiguous()

        # Flatten indices to (B_NH, T*K, 1) and expand to D channels
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        
        # Gather V directly to shape (B_NH, T, K, D_in)
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        # Cast to target dtype for AMP stability
        target_dtype = grad_out.dtype
        v_gathered = v_gathered.to(target_dtype)
        topk_values = topk_values.to(target_dtype)
        weight = weight.to(target_dtype)

        # Pre-compute weighted V (Prime) -> (B_NH, T, K, D_in)
        v_prime = v_gathered * topk_values.unsqueeze(-1) 

        # ---------------------------------------------------------
        # EINSUM BACKWARD PASS (Much cleaner for standard convolution)
        # b: Batch*Heads, t: Sequence, k: Kernel, i: In_Channels, o: Out_Channels
        # ---------------------------------------------------------

        # 1. Gradient w.r.t conv_weight (dW) -> (D_out, D_in, K)
        grad_weight = torch.einsum('bto, btki -> oik', grad_out, v_prime).to(conv_weight.dtype)

        # 2. Gradient w.r.t v_prime (dv_prime) -> (B_NH, T, K, D_in)
        dv_prime = torch.einsum('bto, oik -> btki', grad_out, weight)

        # 3. Gradient w.r.t topk_values (dVal) -> (B_NH, T, K)
        grad_val = (dv_prime * v_gathered).sum(dim=-1).to(topk_values.dtype)

        # 4. Gradient w.r.t V (dV) before scattering
        dv_gathered_final = dv_prime * topk_values.unsqueeze(-1)
        
        # Ensure grad_v strictly matches target_dtype
        grad_v = torch.zeros_like(v, dtype=target_dtype)
        
        # Scatter add gradients back to original V locations
        grad_v.scatter_add_(1, idx_flat, dv_gathered_final.view(B_NH, T * K, D))

        # topk_indices is discrete, so its gradient is None.
        return grad_v, None, grad_val, grad_weight


# ==========================================
# 3. PYTORCH MODULE
# ==========================================
class FastMultiHeadConvNNAttentionStandard(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K, 
                 convolution_type="standard",
                 seq_length=197):

        super(FastMultiHeadConvNNAttentionStandard, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        # STANDARD Convolution weights: (out_channels, in_channels, kernel_size)
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, self.d_k, self.K))

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL)
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Top-K Selection
        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        # Merge Batch and Heads for the Triton Kernel
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # Apply Fused Triton Operation
        out = FusedPrimeConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )

        # Reshape back
        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        # Final Linear Projection
        output = self.W_o(out)
        return output


# ==========================================
# 4. QUICK VERIFICATION TEST
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if device.type != "cuda":
        print("Warning: Triton requires a CUDA-enabled GPU. This test will fail on CPU.")
    else:
        BATCH_SIZE = 2
        SEQ_LENGTH = 197
        D_HIDDEN = 768
        NUM_HEADS = 12
        K = 8
        
        print("Initializing FastMultiHeadConvNNAttention (Standard)...")
        model = FastMultiHeadConvNNAttentionStandard(
            d_hidden=D_HIDDEN, 
            num_heads=NUM_HEADS, 
            attention_dropout=0.1, 
            K=K, 
            seq_length=SEQ_LENGTH
        ).to(device)
        
        x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)
        
        print("Running Forward Pass...")
        out = model(x)
        print(f"Output shape: {out.shape} (Expected: [{BATCH_SIZE}, {SEQ_LENGTH}, {D_HIDDEN}])")
        
        print("Running Backward Pass...")
        loss = out.sum()
        loss.backward()
        print(f"Input gradient shape: {x.grad.shape}")
        print("Success! Forward and backward passes completed.")

Initializing FastMultiHeadConvNNAttention (Standard)...
Running Forward Pass...
Output shape: torch.Size([2, 197, 768]) (Expected: [2, 197, 768])
Running Backward Pass...
Input gradient shape: torch.Size([2, 197, 768])
Success! Forward and backward passes completed.


In [37]:
import torch

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above)

def verify_correctness():
    # Force PyTorch to use strict FP32 instead of TF32 Tensor Cores
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='standard', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttentionStandard(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to Triton model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Triton implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to Triton model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000381

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000143
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000763
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00001717

SUCCESS: Triton implementation is mathematically equivalent!


# ** Modular implementation Test **

In [38]:
"""
FastConvNNAttention.py

- Implementation of ConvNNAttention with Depthwise Convolution using Triton for fused operations.
- Fuses the gathering of V, multiplication by Top-K attention weights, and the depthwise convolution step into a single efficient kernel.
"""

import torch
import torch.nn as nn
import numpy as np
import triton
import triton.language as tl
from torch.amp import custom_fwd, custom_bwd

"""Depthwise Convolution Version of the Fused Attention-ConvNNAttention Kernel."""
@triton.jit
def fused_prime_depthwise_conv_fwd_kernel(
    # Pointers to matrices
    v_ptr, indices_ptr, values_ptr, weight_ptr, out_ptr,
    # Strides to handle memory layout
    stride_vb, stride_vt, stride_vd,
    stride_ib, stride_it, stride_ik,
    stride_wb, stride_wk,
    stride_ob, stride_ot, stride_od,
    # Matrix dimensions
    B_NH, T, D: tl.constexpr, K: tl.constexpr,
    # Meta-parameters
    BLOCK_D: tl.constexpr
):
    """
    Fuses the gathering of V, multiplication by Top-K attention weights, 
    and the depthwise convolution step.
    """
    pid_b_t = tl.program_id(0) # 1D grid covering Batch*Heads and Seq_len
    pid_b = pid_b_t // T
    pid_t = pid_b_t % T

    # Set up channel offsets
    d_offsets = tl.arange(0, BLOCK_D)
    mask_d = d_offsets < D

    # Initialize accumulator for the convolution sum
    acc = tl.zeros([BLOCK_D], dtype=tl.float32)

    # Loop over the Top-K elements
    for k in range(K):
        # 1. Load the index and attention value for the k-th top element
        idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
        v_idx = tl.load(indices_ptr + idx_offset)
        attn_val = tl.load(values_ptr + idx_offset)

        # 2. Load the V vector for all D channels at the gathered index
        v_offsets = pid_b * stride_vb + v_idx * stride_vt + d_offsets * stride_vd
        v_vec = tl.load(v_ptr + v_offsets, mask=mask_d, other=0.0)

        # 3. Load the Depthwise Convolution weight for this K step
        w_offsets = d_offsets * stride_wb + k * stride_wk
        w_vec = tl.load(weight_ptr + w_offsets, mask=mask_d, other=0.0)

        # 4. Multiply and accumulate (Gather * Attn_Value * Conv_Weight)
        acc += v_vec * attn_val * w_vec
        
    # Cast the fp32 accumulator back to the pointer's original dtype (fp16, bf16, or fp32)
    acc_casted = acc.to(v_ptr.dtype.element_ty) 
    
    # Store the final convolved output
    out_offsets = pid_b * stride_ob + pid_t * stride_ot + d_offsets * stride_od
    tl.store(out_ptr + out_offsets, acc_casted, mask=mask_d)

class FusedPrimeDepthwiseConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda') # Tell AMP to let this pass through natively
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        # Save tensors needed for the backward pass
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        
        # Ensure contiguous memory for predictable strides
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.squeeze(1).contiguous() # Squeeze from (D, 1, K) to (D, K)
        
        out = torch.empty_like(v)
        
        # Grid computes one block per query token per batch/head
        grid = lambda meta: (B_NH * T, )
        BLOCK_D = triton.next_power_of_2(D)
        
        fused_prime_depthwise_conv_fwd_kernel[grid](
            v, topk_indices, topk_values, weight, out,
            v.stride(0), v.stride(1), v.stride(2),
            topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
            weight.stride(0), weight.stride(1),
            out.stride(0), out.stride(1), out.stride(2),
            B_NH, T, D, K,
            BLOCK_D=BLOCK_D
        )
        return out

    @staticmethod
    @custom_bwd(device_type='cuda') # Tells AMP how to handle the backward pass
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) # (D, K)
        
        grad_out = grad_out.contiguous()

        # Flatten indices to (B_NH, T*K, 1) and expand to D channels
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        
        # Gather V directly to shape (B_NH, T*K, D), then reshape
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        # Pre-compute shapes for broadcasting and CAST to grad_out's dtype (fp16/bf16)
        target_dtype = grad_out.dtype
        grad_out_exp = grad_out.unsqueeze(2)                
        val_exp = topk_values.unsqueeze(-1).to(target_dtype)                 
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0).to(target_dtype) 
        v_gathered = v_gathered.to(target_dtype)

        # Gradient w.r.t topk_values (dVal) - Cast back to original dtype (fp32)
        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1).to(topk_values.dtype) 

        # Gradient w.r.t conv_weight (dW) - Cast back to original weight dtype (fp32)
        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) 
        grad_weight = grad_weight_raw.t().unsqueeze(1).to(weight.dtype) 

        # Gradient w.r.t V (dV)
        dv_gathered = grad_out_exp * val_exp * weight_t_exp 
        grad_v = torch.zeros_like(v)
        
        # Now both are guaranteed to be target_dtype (e.g., float16)
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        # topk_indices is discrete, so its gradient is None.
        return grad_v, None, grad_val, grad_weight

"""Standard Convolution Version of the Fused Attention-ConvNNAttention Kernel."""
@triton.jit
def fused_prime_standard_conv_fwd_kernel(
    # Pointers to matrices
    v_ptr, indices_ptr, values_ptr, weight_ptr, out_ptr,
    # Strides to handle memory layout
    stride_vb, stride_vt, stride_vd,
    stride_ib, stride_it, stride_ik,
    stride_w_dout, stride_w_din, stride_w_k, # 3 Strides for (D_out, D_in, K)
    stride_ob, stride_ot, stride_od,
    # Matrix dimensions
    B_NH, T, D: tl.constexpr, K: tl.constexpr,
    # Meta-parameters
    BLOCK_D: tl.constexpr
):
    """
    Fuses the gathering of V, multiplication by Top-K attention weights, 
    and a STANDARD 1D convolution step across all channels.
    """
    pid_b_t = tl.program_id(0) # 1D grid covering Batch*Heads and Seq_len
    pid_b = pid_b_t // T
    pid_t = pid_b_t % T

    # Set up channel offsets for both output (d_out) and input (d_in) dimensions
    d_out = tl.arange(0, BLOCK_D)
    d_in = tl.arange(0, BLOCK_D)
    mask_out = d_out < D
    mask_in = d_in < D

    # Initialize accumulator for the output channels
    acc = tl.zeros([BLOCK_D], dtype=tl.float32)

    # Loop over the Top-K elements
    for k in range(K):
        # 1. Load the index and attention value for the k-th top element
        idx_offset = pid_b * stride_ib + pid_t * stride_it + k * stride_ik
        v_idx = tl.load(indices_ptr + idx_offset)
        attn_val = tl.load(values_ptr + idx_offset)

        # 2. Load the Input V vector for all D_in channels at the gathered index
        v_offsets = pid_b * stride_vb + v_idx * stride_vt + d_in * stride_vd
        v_vec = tl.load(v_ptr + v_offsets, mask=mask_in, other=0.0) # Shape: (D_in,)

        # 3. Load the Standard Convolution weight slice for this K step
        w_offsets = d_out[:, None] * stride_w_dout + d_in[None, :] * stride_w_din + k * stride_w_k
        w_slice = tl.load(weight_ptr + w_offsets, mask=(mask_out[:, None] & mask_in[None, :]), other=0.0) # Shape: (D_out, D_in)

        # 4. Multiply and accumulate (Matrix-Vector Product: W @ V * Attn)
        # Broadcasting v_vec across d_out handles the cross-channel mixing
        prod = w_slice * v_vec[None, :] * attn_val
        acc += tl.sum(prod, axis=1) # Sum over D_in (axis 1)
        
    # Cast the fp32 accumulator back to the pointer's original dtype (fp16, bf16, or fp32)
    acc_casted = acc.to(v_ptr.dtype.element_ty) 
    
    # Store the final convolved output
    out_offsets = pid_b * stride_ob + pid_t * stride_ot + d_out * stride_od
    tl.store(out_ptr + out_offsets, acc_casted, mask=mask_out)


# ==========================================
# 2. AUTOGRAD WRAPPER (FORWARD + BACKWARD)
# ==========================================
class FusedPrimeStandardConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda')
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        
        # Ensure contiguous memory for predictable strides
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.contiguous() # (D_out, D_in, K) - No squeezing required!
        
        out = torch.empty_like(v)
        
        # Grid computes one block per query token per batch/head
        grid = lambda meta: (B_NH * T, )
        BLOCK_D = triton.next_power_of_2(D)
        
        fused_prime_standard_conv_fwd_kernel[grid](
            v, topk_indices, topk_values, weight, out,
            v.stride(0), v.stride(1), v.stride(2),
            topk_indices.stride(0), topk_indices.stride(1), topk_indices.stride(2),
            weight.stride(0), weight.stride(1), weight.stride(2), # 3 Strides mapped
            out.stride(0), out.stride(1), out.stride(2),
            B_NH, T, D, K,
            BLOCK_D=BLOCK_D
        )
        return out

    @staticmethod
    @custom_bwd(device_type='cuda') 
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight # (D_out, D_in, K)
        
        grad_out = grad_out.contiguous()

        # Flatten indices to (B_NH, T*K, 1) and expand to D channels
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        
        # Gather V directly to shape (B_NH, T, K, D_in)
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        # Cast to target dtype for AMP stability
        target_dtype = grad_out.dtype
        v_gathered = v_gathered.to(target_dtype)
        topk_values = topk_values.to(target_dtype)
        weight = weight.to(target_dtype)

        # Pre-compute weighted V (Prime) -> (B_NH, T, K, D_in)
        v_prime = v_gathered * topk_values.unsqueeze(-1) 

        # ---------------------------------------------------------
        # EINSUM BACKWARD PASS (Much cleaner for standard convolution)
        # b: Batch*Heads, t: Sequence, k: Kernel, i: In_Channels, o: Out_Channels
        # ---------------------------------------------------------

        # 1. Gradient w.r.t conv_weight (dW) -> (D_out, D_in, K)
        grad_weight = torch.einsum('bto, btki -> oik', grad_out, v_prime).to(conv_weight.dtype)

        # 2. Gradient w.r.t v_prime (dv_prime) -> (B_NH, T, K, D_in)
        dv_prime = torch.einsum('bto, oik -> btki', grad_out, weight)

        # 3. Gradient w.r.t topk_values (dVal) -> (B_NH, T, K)
        grad_val = (dv_prime * v_gathered).sum(dim=-1).to(topk_values.dtype)

        # 4. Gradient w.r.t V (dV) before scattering
        dv_gathered_final = dv_prime * topk_values.unsqueeze(-1)
        
        # Ensure grad_v strictly matches target_dtype
        grad_v = torch.zeros_like(v, dtype=target_dtype)
        
        # Scatter add gradients back to original V locations
        grad_v.scatter_add_(1, idx_flat, dv_gathered_final.view(B_NH, T * K, D))

        # topk_indices is discrete, so its gradient is None.
        return grad_v, None, grad_val, grad_weight
    
class FastMultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K,
                 convolution_type="depthwise",
                 seq_length=197):

        super(FastMultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length
        self.convolution_type = convolution_type

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        if convolution_type == "standard":
            # Standard Convolution weights (D_out, D_in, K)
            self.conv_weight = nn.Parameter(torch.ones(self.d_k, self.d_k, self.K))
        elif convolution_type == "depthwise":
            # Depthwise Convolution weights matching PyTorch's native Conv1d shape
            self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))
    
    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        # Linear Projection + Split Heads 
        q = self.split_head(self.W_q(x)) # (B, NH, SL, DK)
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Attention Matrix: (B, NH, SL, SL)
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        # Top-K Selection
        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        # Merge Batch and Heads for the Triton Kernel
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # Apply Fused Triton Operation (Forward + Backward handled automatically)
        if self.convolution_type == "standard":
            out = FusedPrimeStandardConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )
        elif self.convolution_type == "depthwise":
            out = FusedPrimeDepthwiseConvFunction.apply(
                v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
            )

        # Reshape back: (B*NH, SL, DK) → (B, NH, SL, DK) → (B, SL, d_hidden)
        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        # Final Linear Projection
        output = self.W_o(out)
        return output


# ==========================================
# 4. QUICK VERIFICATION TEST
# ==========================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if device.type != "cuda":
        print("Warning: Triton requires a CUDA-enabled GPU. This test will fail on CPU.")
    else:
        # Hyperparameters matching standard ViT-Base
        BATCH_SIZE = 2
        SEQ_LENGTH = 197
        D_HIDDEN = 768
        NUM_HEADS = 12
        K = 8
        
        print("Initializing FastMultiHeadConvNNAttention...")
        model = FastMultiHeadConvNNAttention(
            d_hidden=D_HIDDEN, 
            num_heads=NUM_HEADS, 
            attention_dropout=0.1, 
            K=K, 
            seq_length=SEQ_LENGTH, 
            convolution_type="standard" # Change to "standard" to test the standard convolution version
        ).to(device)
        
        # Dummy input
        x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)
        
        print("Running Forward Pass...")
        out = model(x)
        print(f"Output shape: {out.shape} (Expected: [{BATCH_SIZE}, {SEQ_LENGTH}, {D_HIDDEN}])")
        
        print("Running Backward Pass...")
        loss = out.sum()
        loss.backward()
        print(f"Input gradient shape: {x.grad.shape}")
        print("Success! Forward and backward passes completed.")

Initializing FastMultiHeadConvNNAttention...
Running Forward Pass...
Output shape: torch.Size([2, 197, 768]) (Expected: [2, 197, 768])
Running Backward Pass...
Input gradient shape: torch.Size([2, 197, 768])
Success! Forward and backward passes completed.


Depthwise Conv

In [39]:
import torch

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above)

def verify_correctness():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='depthwise', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH, convolution_type='depthwise'
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to Triton model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Triton implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to Triton model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000000

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000015
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000286
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00000238

SUCCESS: Triton implementation is mathematically equivalent!


Standard Conv

In [40]:
import torch

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above)

def verify_correctness():
    # Force PyTorch to use strict FP32 instead of TF32 Tensor Cores
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='standard', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FastMultiHeadConvNNAttentionStandard(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH, convolution_type='standard'
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to Triton model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Triton implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to Triton model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000381

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000167
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000763
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00001717

SUCCESS: Triton implementation is mathematically equivalent!


# CUDA/C++ Implementation Test

In [41]:
!rm -rf /root/.cache/torch_extensions/
!pip install ninja

In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.cpp_extension import load_inline
from torch.amp import custom_fwd, custom_bwd

# ==========================================
# 1. RAW CUDA & C++ SOURCE CODE
# ==========================================
cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

// The CUDA Kernel
template <typename scalar_t>
__global__ void convnn_depthwise_fwd_kernel(
    const scalar_t* __restrict__ v,
    const int64_t* __restrict__ indices,
    const scalar_t* __restrict__ values,
    const scalar_t* __restrict__ weight,
    scalar_t* __restrict__ out,
    int B_NH, int N, int D, int K) {

    // Map thread to channel 'd', block.y to sequence 'n', block.z to batch/head 'b'
    int d = blockIdx.x * blockDim.x + threadIdx.x;
    int n = blockIdx.y;
    int b = blockIdx.z;

    // Ensure we don't read out of bounds if D is not a perfect multiple of blockDim
    if (d < D && n < N && b < B_NH) {
        scalar_t acc = 0.0;

        // Loop over the Top-K neighbors
        for (int k = 0; k < K; ++k) {
            // 1. Calculate offset for indices and values: shape (B_NH, N, K)
            int idx_offset = b * (N * K) + n * K + k;
            int64_t v_idx = indices[idx_offset];
            scalar_t attn_val = values[idx_offset];

            // 2. Calculate offset for V: shape (B_NH, N, D)
            // Conceptually: V[b, v_idx, d]
            int v_offset = b * (N * D) + v_idx * D + d;
            scalar_t v_vec = v[v_offset];

            // 3. Calculate offset for Depthwise Weight: shape (D, 1, K)
            // Conceptually: W[d, 0, k]
            int w_offset = d * K + k;
            scalar_t w_vec = weight[w_offset];

            // 4. Multiply and accumulate
            acc += v_vec * attn_val * w_vec;
        }

        // Write final accumulated value to output: shape (B_NH, N, D)
        int out_offset = b * (N * D) + n * D + d;
        out[out_offset] = acc;
    }
}

// The C++ API to launch the kernel
torch::Tensor convnn_fwd_cuda(
    torch::Tensor v, 
    torch::Tensor indices, 
    torch::Tensor values, 
    torch::Tensor weight) {
    
    // Get dimensions
    int B_NH = v.size(0);
    int N = v.size(1);
    int D = v.size(2);
    int K = indices.size(2);

    // Allocate output tensor
    auto out = torch::empty_like(v);

    // Define execution grid configuration
    const int threads = 128; // Standard warp multiple for memory coalescing
    const int blocks_d = (D + threads - 1) / threads;
    
    dim3 block_dim(threads);
    dim3 grid_dim(blocks_d, N, B_NH);

    // Launch the kernel dynamically matching the input precision (fp32, fp16, bf16)
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(v.scalar_type(), "convnn_depthwise_fwd_kernel", ([&] {
        convnn_depthwise_fwd_kernel<scalar_t><<<grid_dim, block_dim>>>(
            v.data_ptr<scalar_t>(),
            indices.data_ptr<int64_t>(),
            values.data_ptr<scalar_t>(),
            weight.data_ptr<scalar_t>(),
            out.data_ptr<scalar_t>(),
            B_NH, N, D, K
        );
    }));

    return out;
}
"""

cpp_source = """
torch::Tensor convnn_fwd_cuda(torch::Tensor v, torch::Tensor indices, torch::Tensor values, torch::Tensor weight);
"""

# # Compile and load the extension
# efficient_convnn_ext = load_inline(
#     name='efficient_convnn',
#     cpp_sources=cpp_source,
#     cuda_sources=cuda_source,
#     functions=['convnn_fwd_cuda'],
#     with_cuda=True,
#     extra_cflags=['-O3'],
#     extra_cuda_cflags=['-O3']
# )


import os

# Create a local build directory we can see
build_dir = "./custom_kernel_build"
os.makedirs(build_dir, exist_ok=True)

# Compile and load the extension
efficient_convnn_ext = load_inline(
    name='efficient_convnn',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['convnn_fwd_cuda'],
    with_cuda=True,
    extra_cflags=['-O3'],
    extra_cuda_cflags=['-O3', '-allow-unsupported-compiler'], # Sometimes needed on Colab
    build_directory=build_dir,
    verbose=True # <-- THIS IS THE MAGIC BULLET
)

# ==========================================
# 2. AUTOGRAD WRAPPER (Hybrid Architecture)
# ==========================================
class EfficientPrimeConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda')
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        # Ensure contiguous memory to prevent pointer offset miscalculations in C++
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.squeeze(1).contiguous() 
        
        # Call our compiled custom CUDA kernel
        out = efficient_convnn_ext.convnn_fwd_cuda(v, topk_indices, topk_values, weight)
        return out

    @staticmethod
    @custom_bwd(device_type='cuda')
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) 
        
        grad_out = grad_out.contiguous()

        # Hybrid ATen Backward Pass (Avoids H100 atomic contention)
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        target_dtype = grad_out.dtype
        grad_out_exp = grad_out.unsqueeze(2)                
        val_exp = topk_values.unsqueeze(-1).to(target_dtype)                 
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0).to(target_dtype) 
        v_gathered = v_gathered.to(target_dtype)

        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1).to(topk_values.dtype) 

        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) 
        grad_weight = grad_weight_raw.t().unsqueeze(1).to(weight.dtype) 

        dv_gathered = grad_out_exp * val_exp * weight_t_exp 
        grad_v = torch.zeros_like(v, dtype=target_dtype)
        
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        return grad_v, None, grad_val, grad_weight

# ==========================================
# 3. PYTORCH MODULE
# ==========================================
class EfficientMultiHeadConvNNAttention(nn.Module):
    def __init__(self, 
                 d_hidden,
                 num_heads, 
                 attention_dropout, 
                 K,
                 seq_length=197):

        super(EfficientMultiHeadConvNNAttention, self).__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.attention_dropout = attention_dropout 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)

        # Depthwise Convolution weights
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x)) 
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)

        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # Launch the custom C++/CUDA kernel via the Autograd Wrapper
        out = EfficientPrimeConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )

        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        output = self.W_o(out)
        return output

In [43]:
import torch

# (Assume MultiHeadConvNNAttention and EfficientMultiHeadConvNNAttention are defined above)

def verify_correctness():
    # Force PyTorch to use strict FP32 instead of TF32 Tensor Cores
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='depthwise', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = EfficientMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to CUDA model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: CUDA implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to CUDA model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000000

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000015
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000286
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00000238

SUCCESS: CUDA implementation is mathematically equivalent!


In [44]:
# import torch
# import torch.nn as nn
# import numpy as np
# from torch.utils.cpp_extension import load_inline
# from torch.amp import custom_fwd, custom_bwd

# # ==========================================
# # 1. OPTIMIZED SHARED MEMORY CUDA KERNEL
# # ==========================================
# cuda_source = """
# #include <torch/extension.h>
# #include <cuda.h>
# #include <cuda_runtime.h>

# template <typename scalar_t>
# __global__ void convnn_fwd_shared_kernel(
#     const scalar_t* __restrict__ v,
#     const int64_t* __restrict__ indices,
#     const scalar_t* __restrict__ values,
#     const scalar_t* __restrict__ weight,
#     scalar_t* __restrict__ out,
#     int B_NH, int N, int D, int K) {

#     int d = threadIdx.x; // Channel dimension
#     int n = blockIdx.x;  // Sequence token dimension
#     int b = blockIdx.y;  // Batch * Head dimension

#     // Dynamically allocate shared memory (SRAM) for K indices and K values
#     extern __shared__ char shared_mem[];
#     int64_t* s_indices = (int64_t*)shared_mem;
#     scalar_t* s_values = (scalar_t*)&s_indices[K];

#     // COOPERATIVE LOAD: 
#     // Instead of D threads reading the same K elements from slow global memory,
#     // the first K threads load the data into ultra-fast shared memory once.
#     if (d < K) {
#         int offset = b * (N * K) + n * K + d;
#         s_indices[d] = indices[offset];
#         s_values[d] = values[offset];
#     }
    
#     // Barrier: Wait for the K threads to finish loading
#     __syncthreads();

#     // COMPUTE AGGREGATION
#     if (d < D) {
#         scalar_t acc = 0.0;
        
#         #pragma unroll
#         for (int k = 0; k < K; ++k) {
#             // Read from fast shared memory
#             int64_t v_idx = s_indices[k];
#             scalar_t attn_val = s_values[k];

#             // Read V and Weight
#             scalar_t v_vec = v[b * N * D + v_idx * D + d];
#             scalar_t w_vec = weight[d * K + k];

#             acc += v_vec * attn_val * w_vec;
#         }
#         out[b * N * D + n * D + d] = acc;
#     }
# }

# torch::Tensor convnn_fwd_cuda(
#     torch::Tensor v, 
#     torch::Tensor indices, 
#     torch::Tensor values, 
#     torch::Tensor weight) {
    
#     int B_NH = v.size(0);
#     int N = v.size(1);
#     int D = v.size(2);
#     int K = indices.size(2);

#     auto out = torch::empty_like(v);

#     // Thread block covers the channel dimension (D)
#     int threads = D; 
    
#     // Grid covers Sequence (N) and Batch*Heads (B_NH)
#     dim3 block_dim(threads);
#     dim3 grid_dim(N, B_NH);

#     // Calculate dynamic shared memory size needed per block
#     size_t shared_mem_size = (K * sizeof(int64_t)) + (K * sizeof(float));

#     AT_DISPATCH_FLOATING_TYPES_AND_HALF(v.scalar_type(), "convnn_fwd_shared_kernel", ([&] {
#         convnn_fwd_shared_kernel<scalar_t><<<grid_dim, block_dim, shared_mem_size>>>(
#             v.data_ptr<scalar_t>(),
#             indices.data_ptr<int64_t>(),
#             values.data_ptr<scalar_t>(),
#             weight.data_ptr<scalar_t>(),
#             out.data_ptr<scalar_t>(),
#             B_NH, N, D, K
#         );
#     }));

#     return out;
# }
# """

# cpp_source = """
# torch::Tensor convnn_fwd_cuda(torch::Tensor v, torch::Tensor indices, torch::Tensor values, torch::Tensor weight);
# """

# import os
# build_dir = "./custom_kernel_build"
# os.makedirs(build_dir, exist_ok=True)

# efficient_convnn_ext = load_inline(
#     name='efficient_convnn_v2',
#     cpp_sources=cpp_source,
#     cuda_sources=cuda_source,
#     functions=['convnn_fwd_cuda'],
#     with_cuda=True,
#     extra_cflags=['-O3'],
#     extra_cuda_cflags=['-O3', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__'],
#     build_directory=build_dir
# )

# # ==========================================
# # 2. AUTOGRAD WRAPPER 
# # ==========================================
# class EfficientPrimeConvFunction(torch.autograd.Function):
#     @staticmethod
#     @custom_fwd(device_type='cuda')
#     def forward(ctx, v, topk_indices, topk_values, conv_weight):
#         ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
#         v = v.contiguous()
#         topk_indices = topk_indices.contiguous()
#         topk_values = topk_values.contiguous()
#         weight = conv_weight.squeeze(1).contiguous() 
        
#         out = efficient_convnn_ext.convnn_fwd_cuda(v, topk_indices, topk_values, weight)
#         return out

#     @staticmethod
#     @custom_bwd(device_type='cuda')
#     def backward(ctx, grad_out):
#         v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
#         B_NH, T, D = v.shape
#         _, _, K = topk_indices.shape
#         weight = conv_weight.squeeze(1) 
        
#         grad_out = grad_out.contiguous()

#         idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
#         v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

#         target_dtype = grad_out.dtype
#         grad_out_exp = grad_out.unsqueeze(2)                
#         val_exp = topk_values.unsqueeze(-1).to(target_dtype)                 
#         weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0).to(target_dtype) 
#         v_gathered = v_gathered.to(target_dtype)

#         grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1).to(topk_values.dtype) 
#         grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) 
#         grad_weight = grad_weight_raw.t().unsqueeze(1).to(weight.dtype) 

#         dv_gathered = grad_out_exp * val_exp * weight_t_exp 
#         grad_v = torch.zeros_like(v, dtype=target_dtype)
#         grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

#         return grad_v, None, grad_val, grad_weight

# # ==========================================
# # 3. CHUNKED PYTORCH MODULE
# # ==========================================
# class FastMultiHeadConvNNAttention(nn.Module):
#     def __init__(self, d_hidden, num_heads, attention_dropout, K, seq_length=197):
#         super().__init__()
#         assert d_hidden % num_heads == 0
        
#         self.d_hidden = d_hidden 
#         self.num_heads = num_heads 
#         self.d_k = d_hidden // num_heads 
#         self.K = K 
#         self.seq_length = seq_length
#         self.chunk_size = 256 # Adjust this based on sequence length

#         self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
#         self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
#         self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
#         self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

#         self.dropout = nn.Dropout(attention_dropout)
#         self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

#     def split_head(self, x):
#         B, N, D = x.size() 
#         return x.view(B, N, self.num_heads, self.d_k).transpose(1, 2)

#     def forward(self, x):
#         B = x.shape[0]

#         q = self.split_head(self.W_q(x)) 
#         k = self.split_head(self.W_k(x))
#         v = self.split_head(self.W_v(x))

#         scale = np.sqrt(self.d_k)
        
#         # CHUNKED QK-TopK FUSION
#         # Eliminates the massive N x N allocation wall
#         topk_vals = []
#         topk_inds = []
        
#         for i in range(0, self.seq_length, self.chunk_size):
#             end = min(i + self.chunk_size, self.seq_length)
#             q_chunk = q[:, :, i:end, :]
            
#             # Compute chunked attention
#             attn_chunk = torch.matmul(q_chunk, k.transpose(-2, -1)) / scale
            
#             # Sort only the chunk
#             vals, inds = torch.topk(attn_chunk, k=self.K, dim=-1, largest=True)
#             topk_vals.append(vals)
#             topk_inds.append(inds)

#         # Reassemble
#         topk_values = torch.cat(topk_vals, dim=2)
#         topk_indices = torch.cat(topk_inds, dim=2)
#         topk_values = torch.softmax(topk_values, dim=-1)

#         # Merge for CUDA Kernel
#         v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
#         topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
#         topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

#         out = EfficientPrimeConvFunction.apply(
#             v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
#         )

#         out = out.view(B, self.num_heads, self.seq_length, self.d_k)
#         out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        
#         return self.W_o(self.dropout(out))

In [45]:
import torch
import numpy as np

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above this)

def benchmark_module(module, x, num_iters=100):
    # 1. Warm-up
    # GPUs have initialization overhead. We run a few dummy passes first.
    for _ in range(10):
        out = module(x)
        loss = out.sum()
        loss.backward()
    
    torch.cuda.synchronize()
    
    # 2. Memory Benchmark
    # Forward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    out = module(x)
    fwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # Backward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    loss = out.sum()
    loss.backward()
    bwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # 3. Speed Benchmark using CUDA Events
    fwd_times = []
    bwd_times = []
    
    for _ in range(num_iters):
        # Time Forward
        torch.cuda.synchronize()
        start_fwd = torch.cuda.Event(enable_timing=True)
        end_fwd = torch.cuda.Event(enable_timing=True)
        
        start_fwd.record()
        out = module(x)
        end_fwd.record()
        torch.cuda.synchronize()
        fwd_times.append(start_fwd.elapsed_time(end_fwd))
        
        # Time Backward
        loss = out.sum()
        torch.cuda.synchronize()
        start_bwd = torch.cuda.Event(enable_timing=True)
        end_bwd = torch.cuda.Event(enable_timing=True)
        
        start_bwd.record()
        loss.backward()
        end_bwd.record()
        torch.cuda.synchronize()
        bwd_times.append(start_bwd.elapsed_time(end_bwd))
        
    avg_fwd = np.mean(fwd_times)
    avg_bwd = np.mean(bwd_times)
    
    return avg_fwd, avg_bwd, fwd_mem, bwd_mem

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Benchmarking requires a CUDA GPU.")

    # Standard ViT parameters
    BATCH_SIZE = 32
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 9
    
    print(f"Benchmarking with Batch Size: {BATCH_SIZE}, Seq Length: {SEQ_LENGTH}")
    print("-" * 60)

    # Initialize Modules

    standard_model = MultiHeadAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0
    ).to(device)
    
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH, convolution_type='depthwise'
    ).to(device)
    
    triton_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    cuda_model = EfficientMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # We reuse the exact same input tensor to ensure fairness
    x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)

    print("Benchmarking Standard Multi-Head Attention...")
    std_fwd, std_bwd, std_fmem, std_bmem = benchmark_module(standard_model, x)
    
    print("Benchmarking Original Implementation...")
    old_fwd, old_bwd, old_fmem, old_bmem = benchmark_module(old_model, x)

    print("Benchmarking Triton Implementation...")
    triton_fwd, triton_bwd, triton_fmem, triton_bmem = benchmark_module(triton_model, x)

    print("Benchmarking CUDA Implementation...")
    cuda_fwd, cuda_bwd, cuda_fmem, cuda_bmem = benchmark_module(cuda_model, x)

    # Print Results Table
    print("\n" + "=" * 85)
    print("Normalized to Original Implementation")
    print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'CUDA':<12} ")
    print("-" * 85)
    print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {triton_fwd:<12.2f} | {cuda_fwd:<12.2f} ")
    print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {triton_bwd:<12.2f} | {cuda_bwd:<12.2f} ")
    print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {triton_fmem:<12.2f} | {cuda_fmem:<12.2f} ")
    print(f"{'Backward Peak Memory (MB)':<25} | {std_bmem:<12.2f} | {old_bmem:<12.2f} | {triton_bmem:<12.2f} | {cuda_bmem:<12.2f} ")
    print("=" * 85)

    # print("\n" + "=" * 115)
    # print("Normalized to Standard Multi-Head Attention")
    # print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'CUDA':<12} | {'Original vs Standard'} | {'Triton vs Standard'} | {'CUDA vs Standard'}")
    # print("-" * 115)
    
    # print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {triton_fwd:<12.2f} | {cuda_fwd:<12.2f} | {old_fwd/std_fwd:.2f}x slower         | {triton_fwd/std_fwd:.2f}x slower       | {cuda_fwd/std_fwd:.2f}x slower")
    
    # print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {triton_bwd:<12.2f} | {cuda_bwd:<12.2f} | {old_bwd/std_bwd:.2f}x slower         | {triton_bwd/std_bwd:.2f}x slower       | {cuda_bwd/std_bwd:.2f}x slower")
    
    # print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {triton_fmem:<12.2f} | {cuda_fmem:<12.2f} | {old_fmem/std_fmem:.2f}x more           | {triton_fmem/std_fmem:.2f}x more         | {cuda_fmem/std_fmem:.2f}x more")
    # print("=" * 115)
    

Benchmarking with Batch Size: 32, Seq Length: 197
------------------------------------------------------------
Benchmarking Standard Multi-Head Attention...
Benchmarking Original Implementation...
Benchmarking Triton Implementation...
Benchmarking CUDA Implementation...

Normalized to Original Implementation
Metric                    | Standard     | Original     | Triton       | CUDA         
-------------------------------------------------------------------------------------
Forward Time (ms)         | 2.58         | 4.51         | 3.73         | 3.66         
Backward Time (ms)        | 5.63         | 9.75         | 7.11         | 7.08         
Forward Peak Memory (MB)  | 412.45       | 908.05       | 400.27       | 409.89       
Backward Peak Memory (MB) | 452.28       | 1054.36      | 770.81       | 780.42       

Normalized to Standard Multi-Head Attention
Metric                    | Standard     | Original     | Triton       | CUDA         | Original vs Standard | Triton vs Sta

# Flash ConvNN Attention Implementation Test

In [46]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.cpp_extension import load_inline
from torch.amp import custom_fwd, custom_bwd
import os

# ==========================================
# 1. RAW CUDA & C++ SOURCE CODE
# ==========================================
cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <float.h>

#define WARP_SIZE 32
#define MAX_K 32 // Maximum neighbors supported by the register heap

// Helper swapping functions for the Heap and Sort
__device__ __forceinline__ void swap_f(float& a, float& b) { float t = a; a = b; b = t; }
__device__ __forceinline__ void swap_i(int& a, int& b) { int t = a; a = b; b = t; }

// Warp-level reduction sum (Collapse 32 values into Lane 0)
__device__ __forceinline__ float warp_reduce_sum(float val) {
    #pragma unroll
    for (int offset = 16; offset > 0; offset /= 2)
        val += __shfl_down_sync(0xffffffff, val, offset);
    return val;
}

// A Min-Heap maintained entirely in ultra-fast hardware registers
struct MinHeap {
    float scores[MAX_K];
    int indices[MAX_K];
    int K;

    __device__ MinHeap(int k) : K(k) {
        #pragma unroll
        for (int i = 0; i < MAX_K; ++i) {
            scores[i] = -FLT_MAX;
            indices[i] = -1;
        }
    }

    __device__ void insert(float score, int idx) {
        if (score > scores[0]) {
            scores[0] = score;
            indices[0] = idx;
            
            int curr = 0;
            while (true) {
                int left = 2 * curr + 1;
                int right = 2 * curr + 2;
                int smallest = curr;

                if (left < K && scores[left] < scores[smallest]) smallest = left;
                if (right < K && scores[right] < scores[smallest]) smallest = right;
                
                if (smallest != curr) {
                    swap_f(scores[curr], scores[smallest]);
                    swap_i(indices[curr], indices[smallest]);
                    curr = smallest;
                } else {
                    break;
                }
            }
        }
    }
};

template <typename scalar_t>
__global__ void flash_convnn_fwd_kernel(
    const scalar_t* __restrict__ Q,
    const scalar_t* __restrict__ K_tensor,
    const scalar_t* __restrict__ V,
    const scalar_t* __restrict__ W,
    scalar_t* __restrict__ Out,
    scalar_t* __restrict__ Out_Vals,
    int64_t* __restrict__ Out_Inds,
    float scale, int B_NH, int N, int D, int K_neighbors) {

    // 1 Warp (32 threads) processes 1 Query token
    int warp_id = threadIdx.y;  // e.g., 0 to 3
    int lane_id = threadIdx.x;  // 0 to 31
    int q_idx = blockIdx.x * blockDim.y + warp_id;
    int b_nh = blockIdx.y;

    if (q_idx >= N) return;

    // Shared memory for streaming Keys (Size: BLOCK_KV * D)
    extern __shared__ float s_K[]; 
    int BLOCK_KV = 64; 

    // Load local Query into registers. (Supports D=32, 64, 96, 128)
    int d_iters = D / WARP_SIZE;
    float q_local[4] = {0.0f}; 
    
    for (int i = 0; i < d_iters; ++i) {
        q_local[i] = Q[b_nh * (N * D) + q_idx * D + (lane_id + i * WARP_SIZE)];
    }

    MinHeap heap(K_neighbors);
    int tid = warp_id * WARP_SIZE + lane_id;
    int num_threads = blockDim.y * WARP_SIZE;

    // ==========================================
    // FLASH LOOP: Stream K blocks without allocating NxN
    // ==========================================
    for (int kv_start = 0; kv_start < N; kv_start += BLOCK_KV) {
        int num_k_valid = min(BLOCK_KV, N - kv_start);
        int total_elements = num_k_valid * D;
        
        // Cooperative load of K into shared memory
        for (int i = tid; i < total_elements; i += num_threads) {
            s_K[i] = K_tensor[b_nh * (N * D) + kv_start * D + i];
        }
        __syncthreads();

        // Compute dot products and update Heap
        for (int k_idx = 0; k_idx < num_k_valid; ++k_idx) {
            float dot = 0.0f;
            for (int i = 0; i < d_iters; ++i) {
                dot += q_local[i] * s_K[k_idx * D + (lane_id + i * WARP_SIZE)];
            }
            // Warp reduction: sum across all 32 threads
            float score = warp_reduce_sum(dot) * scale;
            
            if (lane_id == 0) {
                heap.insert(score, kv_start + k_idx);
            }
        }
        __syncthreads();
    }

    // ==========================================
    // FUSION: Sort, Softmax, and Convolution
    // ==========================================
    if (lane_id == 0) {
        // Bubble Sort the Min-Heap to guarantee correct depthwise weight alignment
        for (int i = 0; i < K_neighbors - 1; ++i) {
            for (int j = 0; j < K_neighbors - i - 1; ++j) {
                if (heap.scores[j] < heap.scores[j+1]) {
                    swap_f(heap.scores[j], heap.scores[j+1]);
                    swap_i(heap.indices[j], heap.indices[j+1]);
                }
            }
        }

        // Softmax Math
        float max_score = heap.scores[0]; 
        float sum_exp = 0.0f;
        for(int k = 0; k < K_neighbors; ++k) {
            if (heap.indices[k] != -1) {
                heap.scores[k] = expf(heap.scores[k] - max_score);
                sum_exp += heap.scores[k];
            }
        }
        for(int k = 0; k < K_neighbors; ++k) {
            heap.scores[k] /= sum_exp;
            // Write Top-K to HBM for PyTorch Autograd Backward Pass
            Out_Vals[b_nh * (N * K_neighbors) + q_idx * K_neighbors + k] = heap.scores[k];
            Out_Inds[b_nh * (N * K_neighbors) + q_idx * K_neighbors + k] = heap.indices[k];
        }
    }

    // Compute Convolution Aggregation
    float final_out[4] = {0.0f};

    for(int k = 0; k < K_neighbors; ++k) {
        // Broadcast the winning index and probability from Lane 0 to the rest of the Warp
        int v_idx = __shfl_sync(0xffffffff, (lane_id == 0 ? heap.indices[k] : 0), 0);
        float prob = __shfl_sync(0xffffffff, (lane_id == 0 ? heap.scores[k] : 0), 0);

        if (v_idx != -1) {
            for (int i = 0; i < d_iters; ++i) {
                int d = lane_id + i * WARP_SIZE;
                float v_val = V[b_nh * (N * D) + v_idx * D + d];
                float w_val = W[d * K_neighbors + k]; 
                final_out[i] += prob * v_val * w_val;
            }
        }
    }

    // Write final convolved output to HBM
    for (int i = 0; i < d_iters; ++i) {
        int d = lane_id + i * WARP_SIZE;
        Out[b_nh * (N * D) + q_idx * D + d] = final_out[i];
    }
}

torch::Tensor flash_convnn_cuda(
    torch::Tensor Q, torch::Tensor K_tensor, torch::Tensor V, torch::Tensor W,
    torch::Tensor out_vals, torch::Tensor out_inds, float scale) {
    
    int B_NH = Q.size(0);
    int N = Q.size(1);
    int D = Q.size(2);
    int K_neighbors = W.size(1);

    auto Out = torch::empty_like(Q);

    int BLOCK_Q = 4; // 4 Queries handled per block
    dim3 block_dim(WARP_SIZE, BLOCK_Q);
    dim3 grid_dim((N + BLOCK_Q - 1) / BLOCK_Q, B_NH);

    int BLOCK_KV = 64;
    size_t shared_mem_size = BLOCK_KV * D * sizeof(float);

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(Q.scalar_type(), "flash_convnn_fwd_kernel", ([&] {
        flash_convnn_fwd_kernel<scalar_t><<<grid_dim, block_dim, shared_mem_size>>>(
            Q.data_ptr<scalar_t>(),
            K_tensor.data_ptr<scalar_t>(),
            V.data_ptr<scalar_t>(),
            W.data_ptr<scalar_t>(),
            Out.data_ptr<scalar_t>(),
            out_vals.data_ptr<scalar_t>(),
            out_inds.data_ptr<int64_t>(),
            scale, B_NH, N, D, K_neighbors
        );
    }));

    return Out;
}
"""

cpp_source = """
torch::Tensor flash_convnn_cuda(
    torch::Tensor Q, torch::Tensor K_tensor, torch::Tensor V, torch::Tensor W,
    torch::Tensor out_vals, torch::Tensor out_inds, float scale);
"""

# Compile the C++ extension (Takes ~2 mins on first run)
build_dir = "./flash_kernel_build"
os.makedirs(build_dir, exist_ok=True)

flash_convnn_ext = load_inline(
    name='flash_convnn',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['flash_convnn_cuda'],
    with_cuda=True,
    extra_cflags=['-O3'],
    extra_cuda_cflags=['-O3', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__'],
    build_directory=build_dir
)

# ==========================================
# 2. AUTOGRAD WRAPPER 
# ==========================================
class FlashPrimeConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda')
    def forward(ctx, q, k, v, conv_weight, scale):
        
        q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
        weight = conv_weight.squeeze(1).contiguous() 
        
        B_NH, N, D = q.shape
        K_neighbors = weight.shape[1]
        
        # Pre-allocate output buffers for the Top-K logic to use in backward pass
        out_vals = torch.empty((B_NH, N, K_neighbors), device=q.device, dtype=q.dtype)
        out_inds = torch.empty((B_NH, N, K_neighbors), device=q.device, dtype=torch.int64)
        
        # Execute Flash Kernel
        out = flash_convnn_ext.flash_convnn_cuda(q, k, v, weight, out_vals, out_inds, scale)
        
        ctx.save_for_backward(v, out_inds, out_vals, conv_weight)
        return out

    @staticmethod
    @custom_bwd(device_type='cuda')
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) 
        
        grad_out = grad_out.contiguous()

        # Hybrid ATen Backward Pass (Maintains memory efficiency without atomic contention)
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        target_dtype = grad_out.dtype
        grad_out_exp = grad_out.unsqueeze(2)                
        val_exp = topk_values.unsqueeze(-1).to(target_dtype)                 
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0).to(target_dtype) 
        v_gathered = v_gathered.to(target_dtype)

        # Gradient computations
        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1).to(topk_values.dtype) 
        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) 
        grad_weight = grad_weight_raw.t().unsqueeze(1).to(weight.dtype) 

        dv_gathered = grad_out_exp * val_exp * weight_t_exp 
        grad_v = torch.zeros_like(v, dtype=target_dtype)
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        # We return Gradients in exact order of inputs (q, k, v, conv_weight, scale)
        # Note: True FlashAttention computes grad_q and grad_k. Because we use Top-K, 
        # gradients do not backpropagate through the discrete sorting step directly here. 
        # If your ConvNN requires Q/K gradients, you must compute the full matrix gradient, 
        # but standard KVT sparse implementations treat the indexing as hard routing.
        return None, None, grad_v, grad_weight, None

# ==========================================
# 3. PYTORCH MODULE
# ==========================================
class FlashMultiHeadConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K, seq_length=197):
        super().__init__()
        
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        self.d_k = d_hidden // num_heads 
        assert self.d_k % 32 == 0, "d_k must be a multiple of 32 for warp parallelism"
        assert K <= 32, "This register-bound kernel supports a maximum K of 32"

        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

    def split_head(self, x):
        B, N, D = x.size() 
        return x.view(B, N, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x)) 
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # Merge Batch and Head dims for the flat Flash kernel
        q_merged = q.reshape(B * self.num_heads, self.seq_length, self.d_k)
        k_merged = k.reshape(B * self.num_heads, self.seq_length, self.d_k)
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)

        scale = float(1.0 / np.sqrt(self.d_k))

        # FLASH FUSION: Everything happens inside the C++ Kernel
        out = FlashPrimeConvFunction.apply(
            q_merged, k_merged, v_merged, self.conv_weight, scale
        )

        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        return self.W_o(out)

In [50]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.cpp_extension import load_inline
from torch.amp import custom_fwd, custom_bwd
import os

# ==========================================
# 1. C++ / CUDA GATHER-FUSION ENGINE
# ==========================================
cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

template <typename scalar_t>
__global__ void exact_gather_conv_fwd_kernel(
    const scalar_t* __restrict__ v,
    const int64_t* __restrict__ indices,
    const scalar_t* __restrict__ values,
    const scalar_t* __restrict__ weight,
    scalar_t* __restrict__ out,
    int B_NH, int N, int D, int K) {

    // Thread maps to Channel (D)
    int d = threadIdx.x; 
    // Block maps to Sequence Token (N) and Batch/Head (B_NH)
    int n = blockIdx.x;  
    int b = blockIdx.y;  

    // Dynamically allocate shared memory (SRAM) for K indices and K values
    extern __shared__ char shared_mem[];
    int64_t* s_indices = (int64_t*)shared_mem;
    scalar_t* s_values = (scalar_t*)&s_indices[K];

    // -----------------------------------------------------------
    // COOPERATIVE LOAD: 
    // The first K threads load the routing data into ultra-fast 
    // shared memory so all D threads don't spam global memory.
    // -----------------------------------------------------------
    if (d < K) {
        int offset = b * (N * K) + n * K + d;
        s_indices[d] = indices[offset];
        s_values[d] = values[offset];
    }
    
    // Barrier: Wait for the K routing parameters to load into SRAM
    __syncthreads();

    // -----------------------------------------------------------
    // AGGREGATION:
    // Compute the V * Softmax * Weight convolution dynamically.
    // -----------------------------------------------------------
    if (d < D) {
        scalar_t acc = 0.0;
        
        #pragma unroll
        for (int k = 0; k < K; ++k) {
            int64_t v_idx = s_indices[k];
            scalar_t attn_val = s_values[k];

            // Direct lookup of V and Weight
            scalar_t v_vec = v[b * N * D + v_idx * D + d];
            scalar_t w_vec = weight[d * K + k];

            acc += v_vec * attn_val * w_vec;
        }
        
        out[b * N * D + n * D + d] = acc;
    }
}

torch::Tensor gather_conv_fwd_cuda(
    torch::Tensor v, 
    torch::Tensor indices, 
    torch::Tensor values, 
    torch::Tensor weight) {
    
    int B_NH = v.size(0);
    int N = v.size(1);
    int D = v.size(2);
    int K = indices.size(2);

    auto out = torch::empty_like(v);

    // Grid configuration: 1 Thread per Channel
    int threads = D; 
    dim3 block_dim(threads);
    dim3 grid_dim(N, B_NH);

    // Calculate dynamic shared memory size needed per sequence token
    size_t shared_mem_size = (K * sizeof(int64_t)) + (K * sizeof(float));

    AT_DISPATCH_FLOATING_TYPES_AND_HALF(v.scalar_type(), "exact_gather_conv_fwd_kernel", ([&] {
        exact_gather_conv_fwd_kernel<scalar_t><<<grid_dim, block_dim, shared_mem_size>>>(
            v.data_ptr<scalar_t>(),
            indices.data_ptr<int64_t>(),
            values.data_ptr<scalar_t>(),
            weight.data_ptr<scalar_t>(),
            out.data_ptr<scalar_t>(),
            B_NH, N, D, K
        );
    }));

    return out;
}
"""

cpp_source = """
torch::Tensor gather_conv_fwd_cuda(torch::Tensor v, torch::Tensor indices, torch::Tensor values, torch::Tensor weight);
"""

# Compile the CUDA Engine
build_dir = "./strategy_b_build"
os.makedirs(build_dir, exist_ok=True)

strategy_b_ext = load_inline(
    name='strategy_b_convnn',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['gather_conv_fwd_cuda'],
    with_cuda=True,
    extra_cflags=['-O3'],
    extra_cuda_cflags=['-O3', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__'],
    build_directory=build_dir
)

# ==========================================
# 2. HYBRID AUTOGRAD WRAPPER
# ==========================================
class ExactGatherConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda')
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        weight = conv_weight.squeeze(1).contiguous() 
        
        # Pass 2: CUDA Gather-Fusion Execution
        out = strategy_b_ext.gather_conv_fwd_cuda(v, topk_indices, topk_values, weight)
        return out

    @staticmethod
    @custom_bwd(device_type='cuda')
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) 
        
        grad_out = grad_out.contiguous()

        # Hybrid ATen scatter_add_ to bypass backward atomic contention
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        target_dtype = grad_out.dtype
        grad_out_exp = grad_out.unsqueeze(2)                
        val_exp = topk_values.unsqueeze(-1).to(target_dtype)                 
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0).to(target_dtype) 
        v_gathered = v_gathered.to(target_dtype)

        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1).to(topk_values.dtype) 
        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) 
        grad_weight = grad_weight_raw.t().unsqueeze(1).to(weight.dtype) 

        dv_gathered = grad_out_exp * val_exp * weight_t_exp 
        grad_v = torch.zeros_like(v, dtype=target_dtype)
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        return grad_v, None, grad_val, grad_weight

# ==========================================
# 3. CONV-NN ATTENTION MODULE
# ==========================================
class StrategyB_ConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K, seq_length=197):
        super().__init__()
        assert d_hidden % num_heads == 0
        
        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)
        # Depthwise configuration
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

    def split_head(self, x):
        B, N, D = x.size() 
        return x.view(B, N, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x)) 
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # PASS 1: The Routing Pass (Exact Global Top-K via Native PyTorch)
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        # Exact sorting is preserved, allowing gradients to flow to W_q and W_k perfectly
        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        # Merge dimensions for the flat C++ kernel
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # PASS 2: The Gather-Fusion Pass (Custom CUDA)
        out = ExactGatherConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )

        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        return self.W_o(out)

In [54]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.cpp_extension import load_inline
from torch.amp import custom_fwd, custom_bwd
import os

# =========================================================================
# 1. C++ / CUDA GATHER-FUSION ENGINE (PASS 2: AGGREGATION & CONVOLUTION)
# =========================================================================
# This CUDA kernel completely eliminates the O(B*H*N*K*D) memory allocation 
# wall by fusing the gather and depthwise convolution into a single efficient
# operation. It cooperatively loads the Top-K indices and values into ultra-fast
# shared memory (SRAM) once per query token, dramatically reducing memory bandwidth
# pressure on High Bandwidth Memory (HBM).

cuda_source = """
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

template <typename scalar_t>
__global__ void exact_gather_conv_fwd_kernel(
    const scalar_t* __restrict__ v,
    const int64_t* __restrict__ indices,
    const scalar_t* __restrict__ values,
    const scalar_t* __restrict__ weight,
    scalar_t* __restrict__ out,
    int B_NH, int T, int D, int K) {

    // Thread ID maps to the channel dimension (D)
    int d = threadIdx.x; 
    // Block ID maps to the sequence token (T) and the Batch*Head (B_NH) dimensions
    int t = blockIdx.x;  
    int b = blockIdx.y;  

    // Ensure we don't calculate out of bounds if D isn't a perfect multiple of block size
    if (d >= D) return;

    // Dynamically allocate shared memory (SRAM) for K indices and K values
    extern __shared__ char shared_mem[];
    int64_t* s_indices = (int64_t*)shared_mem;
    scalar_t* s_values = (scalar_t*)&s_indices[K];

    // -----------------------------------------------------------
    // COOPERATIVE LOAD: 
    // The first K threads cooperatively load the Top-K indices and values
    // into ultra-fast shared memory, so all D threads can read them from SRAM.
    // -----------------------------------------------------------
    if (d < K) {
        int offset = b * (T * K) + t * K + d;
        s_indices[d] = indices[offset];
        s_values[d] = values[offset];
    }
    
    // Barrier: Wait for all K routing parameters to load into shared memory
    __syncthreads();

    // -----------------------------------------------------------
    // AGGREGATION & CONVOLUTION:
    // Compute the V[Top-K] * Softmax_Val * Weight convolution dynamically
    // for this channel d, fetching only the specific V vectors needed.
    // -----------------------------------------------------------
    scalar_t acc = 0.0;
    
    #pragma unroll
    for (int k = 0; k < K; ++k) {
        int64_t v_idx = s_indices[k];
        scalar_t attn_val = s_values[k];

        // Ensure we handle invalid indices (e.g., initialized/masked out)
        if (v_idx != -1) {
            // Direct lookup of V and Weight: V[b_nh, v_idx, d], W[d, 0, k]
            scalar_t v_vec = v[b * T * D + v_idx * D + d];
            scalar_t w_vec = weight[d * K + k];

            // Multiply and accumulate
            acc += v_vec * attn_val * w_vec;
        }
    }
    
    // Write final accumulated value to output: Out[b_nh, t, d]
    out[b * T * D + t * D + d] = acc;
}

torch::Tensor gather_conv_fwd_cuda(
    torch::Tensor v, 
    torch::Tensor indices, 
    torch::Tensor values, 
    torch::Tensor weight) {
    
    int B_NH = v.size(0);
    int T = v.size(1);
    int D = v.size(2);
    int K = indices.size(2);

    auto out = torch::empty_like(v);

    // Grid configuration: 1 Thread per Channel (e.g., 64, 128)
    int threads = D; 
    dim3 block_dim(threads);
    // Grid: [Sequence Length T, Batch*Num_Heads B_NH]
    dim3 grid_dim(T, B_NH);

    // Calculate total dynamic shared memory size needed per sequence token
    size_t shared_mem_size = (K * sizeof(int64_t)) + (K * sizeof(float));

    // Launch the kernel dynamically matching the input precision (fp32, fp16, bf16)
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(v.scalar_type(), "exact_gather_conv_fwd_kernel", ([&] {
        exact_gather_conv_fwd_kernel<scalar_t><<<grid_dim, block_dim, shared_mem_size>>>(
            v.data_ptr<scalar_t>(),
            indices.data_ptr<int64_t>(),
            values.data_ptr<scalar_t>(),
            weight.data_ptr<scalar_t>(),
            out.data_ptr<scalar_t>(),
            B_NH, T, D, K
        );
    }));

    return out;
}
"""

cpp_source = """
torch::Tensor gather_conv_fwd_cuda(torch::Tensor v, torch::Tensor indices, torch::Tensor values, torch::Tensor weight);
"""

# Compilation setup
build_dir = "./flash_convnn_build"
os.makedirs(build_dir, exist_ok=True)

# Compile the CUDA Engine (~2 minutes on first run)
strategy_b_ext = load_inline(
    name='flash_convnn_strategy_b',
    cpp_sources=cpp_source,
    cuda_sources=cuda_source,
    functions=['gather_conv_fwd_cuda'],
    with_cuda=True,
    extra_cflags=['-O3'],
    extra_cuda_cflags=['-O3', '-U__CUDA_NO_HALF_OPERATORS__', '-U__CUDA_NO_HALF_CONVERSIONS__'],
    build_directory=build_dir
)

# ==========================================
# 2. HYBRID AUTOGRAD WRAPPER 
# ==========================================
# This wrapper handles the efficient execution and Autograd connection.
# The hybrid backward pass uses the highly optimized ATen scatter_add_ function
# to backpropagate gradients efficiently to V without the bottleneck of atomic additions.
class ExactGatherConvFunction(torch.autograd.Function):
    @staticmethod
    @custom_fwd(device_type='cuda')
    def forward(ctx, v, topk_indices, topk_values, conv_weight):
        # Save tensors needed for the hybrid backward pass
        ctx.save_for_backward(v, topk_indices, topk_values, conv_weight)
        
        # Ensure contiguous memory to prevent pointer offset miscalculations in C++
        v = v.contiguous()
        topk_indices = topk_indices.contiguous()
        topk_values = topk_values.contiguous()
        # W shape is [D, 1, K], need [D, K] for this implementation
        weight = conv_weight.squeeze(1).contiguous() 
        
        # Pass 2 Execution: CUDA Gather-Fusion
        out = strategy_b_ext.gather_conv_fwd_cuda(v, topk_indices, topk_values, weight)
        return out

    @staticmethod
    @custom_bwd(device_type='cuda')
    def backward(ctx, grad_out):
        v, topk_indices, topk_values, conv_weight = ctx.saved_tensors
        B_NH, T, D = v.shape
        _, _, K = topk_indices.shape
        weight = conv_weight.squeeze(1) 
        
        grad_out = grad_out.contiguous()

        # Hybrid ATen scatter_add_ backward pass (Very Efficient)
        idx_flat = topk_indices.view(B_NH, T * K, 1).expand(-1, -1, D)
        v_gathered = torch.gather(v, 1, idx_flat).view(B_NH, T, K, D)

        target_dtype = grad_out.dtype
        grad_out_exp = grad_out.unsqueeze(2)                
        val_exp = topk_values.unsqueeze(-1).to(target_dtype)                 
        weight_t_exp = weight.t().unsqueeze(0).unsqueeze(0).to(target_dtype) 
        v_gathered = v_gathered.to(target_dtype)

        grad_val = (grad_out_exp * v_gathered * weight_t_exp).sum(dim=-1).to(topk_values.dtype) 
        grad_weight_raw = (grad_out_exp * v_gathered * val_exp).sum(dim=(0, 1)) 
        grad_weight = grad_weight_raw.t().unsqueeze(1).to(weight.dtype) 

        dv_gathered = grad_out_exp * val_exp * weight_t_exp 
        grad_v = torch.zeros_like(v, dtype=target_dtype)
        # atomicAdd contamination is avoided here!
        grad_v.scatter_add_(1, idx_flat, dv_gathered.view(B_NH, T * K, D))

        # We return Gradients in the exact order of inputs (v, topk_indices, topk_values, conv_weight)
        return grad_v, None, grad_val, grad_weight

# ==========================================
# 3. CONV-NN ATTENTION MODULE
# ==========================================
class FlashMultiHeadConvNNAttention(nn.Module):
    def __init__(self, d_hidden, num_heads, attention_dropout, K, seq_length=197):
        super().__init__()
        assert d_hidden % num_heads == 0, "d_hidden must be divisible by num_heads"
        
        self.d_hidden = d_hidden 
        self.num_heads = num_heads 
        self.d_k = d_hidden // num_heads 
        self.K = K 
        self.seq_length = seq_length

        # Standard Linear Projections
        self.W_q = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_k = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_v = nn.Linear(d_hidden, d_hidden, bias=False)
        self.W_o = nn.Linear(d_hidden, d_hidden, bias=False)

        self.dropout = nn.Dropout(attention_dropout)
        
        # Depthwise Convolution Weights
        self.conv_weight = nn.Parameter(torch.ones(self.d_k, 1, self.K))

    def split_head(self, x):
        batch_size, seq_length, d_hidden = x.size() 
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x):
        B = x.shape[0]

        q = self.split_head(self.W_q(x)) 
        k = self.split_head(self.W_k(x))
        v = self.split_head(self.W_v(x))

        # PASS 1: The Routing Pass (Exact Global Top-K via Native PyTorch)
        # Using PyTorch native functions guarantees correctness and leverages highly 
        # optimized cuBLAS and parallel Radix Sort kernels under the hood.
        attn_matrix = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        # Exact sorting is preserved, allowing gradients to flow to W_q and W_k perfectly
        topk_values, topk_indices = torch.topk(attn_matrix, k=self.K, dim=-1, largest=True)
        topk_values = torch.softmax(topk_values, dim=-1)

        # Merge dims for the CUDA aggregation kernel
        v_merged = v.reshape(B * self.num_heads, self.seq_length, self.d_k)
        topk_indices_merged = topk_indices.reshape(B * self.num_heads, self.seq_length, self.K)
        topk_values_merged = topk_values.reshape(B * self.num_heads, self.seq_length, self.K)

        # PASS 2: The Gather-Fusion Pass (Custom CUDA Extension)
        out = ExactGatherConvFunction.apply(
            v_merged, topk_indices_merged, topk_values_merged, self.conv_weight
        )

        # Reshape back to standard Multi-Head MHA format and Project Output
        out = out.view(B, self.num_heads, self.seq_length, self.d_k)
        out = out.transpose(1, 2).contiguous().view(B, self.seq_length, self.d_hidden)
        out = self.dropout(out) 

        output = self.W_o(out)
        return output

In [55]:
import torch

# (Assume MultiHeadConvNNAttention and EfficientMultiHeadConvNNAttention are defined above)

def verify_correctness():
    # Force PyTorch to use strict FP32 instead of TF32 Tensor Cores
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='depthwise', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FlashMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to CUDA model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Flash CUDA implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

Copying weights from Original to CUDA model...

Running Forward Passes...
Forward Pass Match: True
Max Forward Difference: 0.00000000

Running Backward Passes...
Input Gradient (x.grad) Match: True
Max Input Gradient Difference: 0.00000017
Conv Weight Gradient Match: True
Max Conv Weight Gradient Difference: 0.00000286
W_q Weight Gradient Match: True
Max W_q Weight Gradient Difference: 0.00000238

SUCCESS: Flash CUDA implementation is mathematically equivalent!


In [56]:
import torch
import numpy as np

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above this)

def benchmark_module(module, x, num_iters=100):
    # 1. Warm-up
    # GPUs have initialization overhead. We run a few dummy passes first.
    for _ in range(10):
        out = module(x)
        loss = out.sum()
        loss.backward()
    
    torch.cuda.synchronize()
    
    # 2. Memory Benchmark
    # Forward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    out = module(x)
    fwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # Backward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    loss = out.sum()
    loss.backward()
    bwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # 3. Speed Benchmark using CUDA Events
    fwd_times = []
    bwd_times = []
    
    for _ in range(num_iters):
        # Time Forward
        torch.cuda.synchronize()
        start_fwd = torch.cuda.Event(enable_timing=True)
        end_fwd = torch.cuda.Event(enable_timing=True)
        
        start_fwd.record()
        out = module(x)
        end_fwd.record()
        torch.cuda.synchronize()
        fwd_times.append(start_fwd.elapsed_time(end_fwd))
        
        # Time Backward
        loss = out.sum()
        torch.cuda.synchronize()
        start_bwd = torch.cuda.Event(enable_timing=True)
        end_bwd = torch.cuda.Event(enable_timing=True)
        
        start_bwd.record()
        loss.backward()
        end_bwd.record()
        torch.cuda.synchronize()
        bwd_times.append(start_bwd.elapsed_time(end_bwd))
        
    avg_fwd = np.mean(fwd_times)
    avg_bwd = np.mean(bwd_times)
    
    return avg_fwd, avg_bwd, fwd_mem, bwd_mem

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Benchmarking requires a CUDA GPU.")

    # Standard ViT parameters
    BATCH_SIZE = 32
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 9
    
    print(f"Benchmarking with Batch Size: {BATCH_SIZE}, Seq Length: {SEQ_LENGTH}")
    print("-" * 60)

    # Initialize Modules

    standard_model = MultiHeadAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0
    ).to(device)
    
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH, convolution_type='depthwise'
    ).to(device)
    
    triton_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    cuda_model = EfficientMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    flash_model = FlashMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    two_pass_model = StrategyB_ConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # We reuse the exact same input tensor to ensure fairness
    x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)

    print("Benchmarking Standard Multi-Head Attention...")
    std_fwd, std_bwd, std_fmem, std_bmem = benchmark_module(standard_model, x)
    
    print("Benchmarking Original Implementation...")
    old_fwd, old_bwd, old_fmem, old_bmem = benchmark_module(old_model, x)

    print("Benchmarking Triton Implementation...")
    triton_fwd, triton_bwd, triton_fmem, triton_bmem = benchmark_module(triton_model, x)

    print("Benchmarking CUDA Implementation...")
    cuda_fwd, cuda_bwd, cuda_fmem, cuda_bmem = benchmark_module(cuda_model, x)

    print("Benchmarking Flash CUDA Implementation...")
    flash_fwd, flash_bwd, flash_fmem, flash_bmem = benchmark_module(flash_model, x)

    print("Benchmarking Two-Pass Strategy B Implementation...")
    two_pass_fwd, two_pass_bwd, two_pass_fmem, two_pass_bmem = benchmark_module(two_pass_model, x)

    # Print Results Table
    print("\n" + "=" * 85)
    print("Normalized to Original Implementation")
    print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'CUDA':<12} | {'Flash':<12} | {'Two-Pass':<12} ")
    print("-" * 85)
    print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {triton_fwd:<12.2f} | {cuda_fwd:<12.2f} | {flash_fwd:<12.2f} | {two_pass_fwd:<12.2f} ")
    print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {triton_bwd:<12.2f} | {cuda_bwd:<12.2f} | {flash_bwd:<12.2f} | {two_pass_bwd:<12.2f} ")
    print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {triton_fmem:<12.2f} | {cuda_fmem:<12.2f} | {flash_fmem:<12.2f} | {two_pass_fmem:<12.2f} ")
    print(f"{'Backward Peak Memory (MB)':<25} | {std_bmem:<12.2f} | {old_bmem:<12.2f} | {triton_bmem:<12.２f} | {cuda_bmem:<1２.２f} | {flash_bmem:<1２.２f} | {two_pass_bmem:<1２.２f} ")
    print("=" * 85)

    # print("\n" + "=" * 115)
    # print("Normalized to Standard Multi-Head Attention")
    # print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'CUDA':<12} | {'Original vs Standard'} | {'Triton vs Standard'} | {'CUDA vs Standard'}")
    # print("-" * 115)
    
    # print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {triton_fwd:<12.2f} | {cuda_fwd:<12.2f} | {old_fwd/std_fwd:.2f}x slower         | {triton_fwd/std_fwd:.2f}x slower       | {cuda_fwd/std_fwd:.2f}x slower")
    
    # print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {triton_bwd:<12.2f} | {cuda_bwd:<12.2f} | {old_bwd/std_bwd:.2f}x slower         | {triton_bwd/std_bwd:.2f}x slower       | {cuda_bwd/std_bwd:.2f}x slower")
    
    # print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {triton_fmem:<12.2f} | {cuda_fmem:<12.2f} | {old_fmem/std_fmem:.2f}x more           | {triton_fmem/std_fmem:.2f}x more         | {cuda_fmem/std_fmem:.2f}x more")
    # print("=" * 115)
    

Benchmarking with Batch Size: 32, Seq Length: 197
------------------------------------------------------------
Benchmarking Standard Multi-Head Attention...
Benchmarking Original Implementation...
Benchmarking Triton Implementation...
Benchmarking CUDA Implementation...
Benchmarking Flash CUDA Implementation...
Benchmarking Two-Pass Strategy B Implementation...

Normalized to Original Implementation
Metric                    | Standard     | Original     | Triton       | CUDA         | Flash        | Two-Pass     
-------------------------------------------------------------------------------------
Forward Time (ms)         | 2.56         | 4.50         | 3.72         | 3.67         | 3.64         | 3.64         
Backward Time (ms)        | 5.54         | 9.69         | 7.05         | 7.09         | 7.07         | 7.08         
Forward Peak Memory (MB)  | 531.06       | 1027.29      | 519.17       | 528.17       | 537.52       | 547.22       
Backward Peak Memory (MB) | 570.89       | 

In [ ]:
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <float.h>

#define WARP_SIZE 32
#define MAX_K 16 // Hardcoded maximum K for register allocation

// Helper to swap elements for the heap
__device__ __forceinline__ void swap(float& a, float& b) { float t = a; a = b; b = t; }
__device__ __forceinline__ void swap_idx(int& a, int& b) { int t = a; a = b; b = t; }

// A simple thread-local Min-Heap maintained entirely in hardware registers
struct ThreadLocalTopK {
    float scores[MAX_K];
    int indices[MAX_K];
    int K;

    __device__ ThreadLocalTopK(int k) : K(k) {
        #pragma unroll
        for (int i = 0; i < MAX_K; ++i) {
            scores[i] = -FLT_MAX;
            indices[i] = -1;
        }
    }

    // Insert a new score. If it's bigger than the smallest (which is at index 0), replace and heapify.
    __device__ void insert(float score, int idx) {
        if (score > scores[0]) {
            scores[0] = score;
            indices[0] = idx;
            
            // Sift down to maintain Min-Heap property
            int curr = 0;
            while (true) {
                int left = 2 * curr + 1;
                int right = 2 * curr + 2;
                int smallest = curr;

                if (left < K && scores[left] < scores[smallest]) smallest = left;
                if (right < K && scores[right] < scores[smallest]) smallest = right;
                
                if (smallest != curr) {
                    swap(scores[curr], scores[smallest]);
                    swap_idx(indices[curr], indices[smallest]);
                    curr = smallest;
                } else {
                    break;
                }
            }
        }
    }
};

template <typename scalar_t>
__global__ void flash_convnn_fwd_kernel(
    const scalar_t* __restrict__ Q,
    const scalar_t* __restrict__ K_tensor,
    const scalar_t* __restrict__ V,
    const scalar_t* __restrict__ W,
    scalar_t* __restrict__ Out,
    float scale, int B_NH, int N, int D, int K_neighbors) {

    // 1 Warp (32 threads) processes exactly 1 Query token.
    int warp_id = threadIdx.y; 
    int lane_id = threadIdx.x; // 0 to 31
    
    // Global Query Index
    int q_idx = blockIdx.x * blockDim.y + warp_id;
    int b_nh = blockIdx.y;

    if (q_idx >= N) return;

    // Allocate shared memory for caching chunks of K and V
    extern __shared__ float s_mem[];
    float* s_K = s_mem;               // [BLOCK_KV, D]
    float* s_V = (float*)&s_K[64 * D]; // [BLOCK_KV, D] (Assuming BLOCK_KV=64)

    // Load this warp's specific Query into registers
    float local_q[128]; // Max D=128
    for(int d = lane_id; d < D; d += WARP_SIZE) {
        local_q[d] = Q[b_nh * (N * D) + q_idx * D + d];
    }

    // Initialize the register Min-Heap
    ThreadLocalTopK heap(K_neighbors);

    // ==========================================
    // INNER LOOP: Stream K blocks and update Heap
    // ==========================================
    int BLOCK_KV = 64;
    for (int kv_start = 0; kv_start < N; kv_start += BLOCK_KV) {
        
        // 1. Cooperative Load K block into Shared Memory
        int num_k_valid = min(BLOCK_KV, N - kv_start);
        for (int i = threadIdx.y * WARP_SIZE + lane_id; i < num_k_valid * D; i += blockDim.y * WARP_SIZE) {
            s_K[i] = K_tensor[b_nh * (N * D) + kv_start * D + i];
        }
        __syncthreads();

        // 2. Compute Dot Products (Distributed across the warp)
        for (int k_step = lane_id; k_step < num_k_valid; k_step += WARP_SIZE) {
            float dot = 0.0f;
            for (int d = 0; d < D; ++d) {
                dot += local_q[d] * s_K[k_step * D + d];
            }
            dot *= scale;
            
            // 3. Push to Thread-Local Min-Heap
            heap.insert(dot, kv_start + k_step);
        }
        __syncthreads();
    }

    // ==========================================
    // WARP REDUCTION: Merge 32 heaps into 1
    // ==========================================
    // (In a full implementation, you use __shfl_sync here to find the absolute 
    // Top-K across all 32 threads in the warp. For brevity, assuming lane 0 gathers them).
    
    // ==========================================
    // FUSED CONVOLUTION: Fetch V and Apply W
    // ==========================================
    if (lane_id == 0) {
        // Compute Softmax on the final Top-K elements in the heap
        float max_score = -FLT_MAX;
        for(int k=0; k<K_neighbors; ++k) max_score = max(max_score, heap.scores[k]);
        
        float sum_exp = 0.0f;
        for(int k=0; k<K_neighbors; ++k) {
            heap.scores[k] = expf(heap.scores[k] - max_score);
            sum_exp += heap.scores[k];
        }
        
        // Accumulate Output
        float final_out[128] = {0}; // Max D
        for(int k=0; k<K_neighbors; ++k) {
            float prob = heap.scores[k] / sum_exp;
            int v_idx = heap.indices[k];
            
            if (v_idx != -1) {
                for (int d = 0; d < D; ++d) {
                    float v_val = V[b_nh * (N * D) + v_idx * D + d];
                    float w_val = W[d * K_neighbors + k]; // Depthwise weight
                    final_out[d] += prob * v_val * w_val;
                }
            }
        }
        
        // Write to HBM
        for(int d=0; d < D; ++d) {
            Out[b_nh * (N * D) + q_idx * D + d] = final_out[d];
        }
    }
}

# Claude FlashMultiHeadConvNNAttention Implementation Test

In [ ]:
import torch

# (Assume MultiHeadConvNNAttention and EfficientMultiHeadConvNNAttention are defined above)

def verify_correctness():
    # Force PyTorch to use strict FP32 instead of TF32 Tensor Cores
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Validation requires a CUDA GPU.")

    # Hyperparameters
    BATCH_SIZE = 4
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 8
    
    # 1. Initialize models
    # CRITICAL: Dropout must be 0.0 so both models are deterministic!
    torch.manual_seed(42)
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, convolution_type='depthwise', seq_length=SEQ_LENGTH
    ).to(device)
    
    new_model = FlashMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, 
        K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # 2. Synchronize Weights
    print("Copying weights from Original to CUDA model...")
    with torch.no_grad():
        new_model.W_q.weight.copy_(old_model.W_q.weight)
        new_model.W_k.weight.copy_(old_model.W_k.weight)
        new_model.W_v.weight.copy_(old_model.W_v.weight)
        new_model.W_o.weight.copy_(old_model.W_o.weight)
        # old_model.conv is an nn.Conv1d for depthwise, its weight shape is (d_k, 1, K)
        new_model.conv_weight.copy_(old_model.conv.weight)

    # 3. Create identical inputs (cloned so they track gradients separately)
    x_base = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device)
    x_old = x_base.clone().requires_grad_(True)
    x_new = x_base.clone().requires_grad_(True)

    # ==========================================
    # FORWARD PASS CHECK
    # ==========================================
    print("\nRunning Forward Passes...")
    out_old = old_model(x_old)
    out_new = new_model(x_new)

    fwd_diff = (out_old - out_new).abs().max().item()
    fwd_match = torch.allclose(out_old, out_new, atol=1e-4)
    
    print(f"Forward Pass Match: {fwd_match}")
    print(f"Max Forward Difference: {fwd_diff:.8f}")

    # ==========================================
    # BACKWARD PASS CHECK
    # ==========================================
    print("\nRunning Backward Passes...")
    # Generate a random incoming gradient from the "next layer"
    grad_out = torch.randn_like(out_old)

    out_old.backward(grad_out)
    out_new.backward(grad_out)

    # Check Input Gradients
    x_grad_diff = (x_old.grad - x_new.grad).abs().max().item()
    x_grad_match = torch.allclose(x_old.grad, x_new.grad, atol=1e-4)
    print(f"Input Gradient (x.grad) Match: {x_grad_match}")
    print(f"Max Input Gradient Difference: {x_grad_diff:.8f}")

    # Check Weight Gradients (Convolution)
    conv_grad_diff = (old_model.conv.weight.grad - new_model.conv_weight.grad).abs().max().item()
    conv_grad_match = torch.allclose(old_model.conv.weight.grad, new_model.conv_weight.grad, atol=1e-4)
    print(f"Conv Weight Gradient Match: {conv_grad_match}")
    print(f"Max Conv Weight Gradient Difference: {conv_grad_diff:.8f}")

    # Check Weight Gradients (Linear Projections - Example: W_q)
    wq_grad_diff = (old_model.W_q.weight.grad - new_model.W_q.weight.grad).abs().max().item()
    wq_grad_match = torch.allclose(old_model.W_q.weight.grad, new_model.W_q.weight.grad, atol=1e-4)
    print(f"W_q Weight Gradient Match: {wq_grad_match}")
    print(f"Max W_q Weight Gradient Difference: {wq_grad_diff:.8f}")

    print("\n" + "="*50)
    if fwd_match and x_grad_match and conv_grad_match and wq_grad_match:
        print("SUCCESS: Flash CUDA implementation is mathematically equivalent!")
    else:
        print("WARNING: Divergence detected between implementations.")
    print("="*50)

if __name__ == "__main__":
    verify_correctness()

In [ ]:
import torch
import numpy as np

# (Assume MultiHeadConvNNAttention and FastMultiHeadConvNNAttention are defined above this)

def benchmark_module(module, x, num_iters=100):
    # 1. Warm-up
    # GPUs have initialization overhead. We run a few dummy passes first.
    for _ in range(10):
        out = module(x)
        loss = out.sum()
        loss.backward()
    
    torch.cuda.synchronize()
    
    # 2. Memory Benchmark
    # Forward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    out = module(x)
    fwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # Backward Pass Memory
    torch.cuda.reset_peak_memory_stats()
    loss = out.sum()
    loss.backward()
    bwd_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) # Convert to MB
    
    # 3. Speed Benchmark using CUDA Events
    fwd_times = []
    bwd_times = []
    
    for _ in range(num_iters):
        # Time Forward
        torch.cuda.synchronize()
        start_fwd = torch.cuda.Event(enable_timing=True)
        end_fwd = torch.cuda.Event(enable_timing=True)
        
        start_fwd.record()
        out = module(x)
        end_fwd.record()
        torch.cuda.synchronize()
        fwd_times.append(start_fwd.elapsed_time(end_fwd))
        
        # Time Backward
        loss = out.sum()
        torch.cuda.synchronize()
        start_bwd = torch.cuda.Event(enable_timing=True)
        end_bwd = torch.cuda.Event(enable_timing=True)
        
        start_bwd.record()
        loss.backward()
        end_bwd.record()
        torch.cuda.synchronize()
        bwd_times.append(start_bwd.elapsed_time(end_bwd))
        
    avg_fwd = np.mean(fwd_times)
    avg_bwd = np.mean(bwd_times)
    
    return avg_fwd, avg_bwd, fwd_mem, bwd_mem

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("Benchmarking requires a CUDA GPU.")

    # Standard ViT parameters
    BATCH_SIZE = 32
    SEQ_LENGTH = 197
    D_HIDDEN = 768
    NUM_HEADS = 12
    K = 9
    
    print(f"Benchmarking with Batch Size: {BATCH_SIZE}, Seq Length: {SEQ_LENGTH}")
    print("-" * 60)

    # Initialize Modules

    standard_model = MultiHeadAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0
    ).to(device)
    
    old_model = MultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH, convolution_type='depthwise'
    ).to(device)
    
    triton_model = FastMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    cuda_model = EfficientMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    flash_model = FlashMultiHeadConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    two_pass_model = StrategyB_ConvNNAttention(
        d_hidden=D_HIDDEN, num_heads=NUM_HEADS, attention_dropout=0.0, K=K, seq_length=SEQ_LENGTH
    ).to(device)

    # We reuse the exact same input tensor to ensure fairness
    x = torch.randn(BATCH_SIZE, SEQ_LENGTH, D_HIDDEN, device=device, requires_grad=True)

    print("Benchmarking Standard Multi-Head Attention...")
    std_fwd, std_bwd, std_fmem, std_bmem = benchmark_module(standard_model, x)
    
    print("Benchmarking Original Implementation...")
    old_fwd, old_bwd, old_fmem, old_bmem = benchmark_module(old_model, x)

    print("Benchmarking Triton Implementation...")
    triton_fwd, triton_bwd, triton_fmem, triton_bmem = benchmark_module(triton_model, x)

    print("Benchmarking CUDA Implementation...")
    cuda_fwd, cuda_bwd, cuda_fmem, cuda_bmem = benchmark_module(cuda_model, x)

    print("Benchmarking Flash CUDA Implementation...")
    flash_fwd, flash_bwd, flash_fmem, flash_bmem = benchmark_module(flash_model, x)

    print("Benchmarking Two-Pass Strategy B Implementation...")
    two_pass_fwd, two_pass_bwd, two_pass_fmem, two_pass_bmem = benchmark_module(two_pass_model, x)

    # Print Results Table
    print("\n" + "=" * 85)
    print("Normalized to Original Implementation")
    print(f"{'Metric':<25} | {'Standard':<12} | {'Original':<12} | {'Triton':<12} | {'CUDA':<12} | {'Flash':<12} | {'Two-Pass':<12} ")
    print("-" * 85)
    print(f"{'Forward Time (ms)':<25} | {std_fwd:<12.2f} | {old_fwd:<12.2f} | {triton_fwd:<12.2f} | {cuda_fwd:<12.2f} | {flash_fwd:<12.2f} | {two_pass_fwd:<12.2f} ")
    print(f"{'Backward Time (ms)':<25} | {std_bwd:<12.2f} | {old_bwd:<12.2f} | {triton_bwd:<12.2f} | {cuda_bwd:<12.2f} | {flash_bwd:<12.2f} | {two_pass_bwd:<12.2f} ")
    print(f"{'Forward Peak Memory (MB)':<25} | {std_fmem:<12.2f} | {old_fmem:<12.2f} | {triton_fmem:<12.2f} | {cuda_fmem:<12.2f} | {flash_fmem:<12.2f} | {two_pass_fmem:<12.2f} ")
    print(f"{'Backward Peak Memory (MB)':<25} | {std_bmem:<12.2f} | {old_bmem:<12.2f} | {triton_bmem:<12.２f} | {cuda_bmem:<1２.２f} | {flash_bmem:<1２.２f} | {two_pass_bmem:<1２.２f} ")
    print("=" * 85)
